# Implementation

## Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from joblib import Parallel, delayed
import csv
import os
import time

np.random.seed(42)

## Functions

In [ ]:
def grid_mask(X, Y, p, D):

    """
    Creates a periodic binary grid mask with circular holes.

    Parameters
    ----------
    X, Y : ndarray
        Coordinate grids, typically generated with np.meshgrid.

    p : float
        Grid period.

    D : float
        Diameter of the circular holes.

    Returns
    -------
    material : ndarray
        Binary grid mask with value 1 for grid material and
        value 0 for holes.
    """

    x_mod = np.mod(X, p) - p/2
    y_mod = np.mod(Y, p) - p/2

    r2 = x_mod**2 + y_mod**2
    hole = r2 < (D/2)**2
    
    material = ~hole

    return material.astype(float)

In [ ]:
def simulation(N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0, d = 10.0, dx = 1.0, dy = 0.0, theta = np.deg2rad(45), 
    phi = np.deg2rad(0), L0 = 1.0, rho_f = 1.0, rho_r = 1.0, Ibg = 0.1, sigma = 0, noise_switch = False, 
    poisson_scale = 1000, gaussian_std = 0.01, gamma_switch = False, gamma = 0.8, mode = "simulation"):

    """
    Simulates the image formation of two parallel periodic grids
    with a relative lateral displacement and view-dependent
    projection shift.

    Parameters
    ----------
    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    d : float
        Distance between the front and rear grids in mm.

    dx, dy : float
        Relative lateral displacement of the rear grid with
        respect to the front grid in mm.

    theta : float
        Polar viewing angle in radians.

    phi : float
        Azimuth viewing angle in radians.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the
        front and rear grids.

    Ibg : float
        Constant background intensity.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, Poisson and additive Gaussian noise are applied
        to the blurred image.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.
        Larger values correspond to lower relative Poisson noise.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied to the final
        intensity image.

    gamma : float
        Gamma exponent used for gamma correction.

    mode : {"simulation", "inverse"}
        Output mode.

        If "simulation", diagnostic plots and shift information
        are displayed in addition to returning the result
        dictionary.

        If "inverse", no diagnostic plots are generated and only
        the result dictionary is returned.

    Returns
    -------
    result : dict
        Dictionary containing the simulated image data and
        intermediate quantities with the following entries:

        "intensity" : ndarray
            Final intensity image after blur, optional noise and
            optional gamma correction.

        "I_ideal" : ndarray
            Ideal intensity image before blur and noise.

        "I_blur" : ndarray
            Intensity image after Gaussian blur.

        "I_final" : ndarray
            Intensity image after optional noise but before gamma
            correction.

        "Of" : ndarray
            Binary mask of the front grid.

        "Or" : ndarray
            Binary mask of the unshifted rear grid.
            Returned only in simulation mode.

        "Or_shift" : ndarray
            Binary mask of the shifted rear grid.

        "Vr" : ndarray
            Visible rear-grid contribution through the front-grid
            holes.

        "sv" : ndarray
            View-dependent lateral projection shift in mm.

        "delta_x", "delta_y" : float
            Effective relative shift of the rear grid in mm.

        "image" : ndarray
            Normalized RGB overlay used for visualization.
    """

    x = np.linspace(-size / 2, size / 2, N)
    y = np.linspace(-size / 2, size / 2, N)

    X, Y = np.meshgrid(x, y)

    Of = grid_mask(X, Y, p, Df)
    Or = grid_mask(X, Y, p, Dr)


    sv = d * np.tan(theta) * np.array([
        np.cos(phi),
        np.sin(phi)
    ])

    delta_x = sv[0] - dx
    delta_y = sv[1] - dy

    Or_shift = grid_mask(X - delta_x, Y - delta_y, p, Dr)

    Vr = (1 - Of) * Or_shift


    I_f = L0 * rho_f * Of
    I_r = L0 * rho_r * Vr

    I_ideal = Ibg + I_f + I_r


    I_blur = gaussian_filter(I_ideal, sigma = sigma)


    if noise_switch:
        I_noise = np.random.poisson(I_blur * poisson_scale) / poisson_scale
        I_noise += np.random.normal(0, gaussian_std, I_ideal.shape)
        I_noise = np.clip(I_noise, 0, None)

        I_final = I_noise

    else:
        I_final = I_blur

    if gamma_switch:
        I_gamma = np.power(I_final, gamma)

    else:
        I_gamma = I_final

    I_rgb = np.zeros((N, N, 3))

    I_rgb[:, :, 0] = I_gamma * (Vr > 0)
    I_rgb[:, :, 2] = I_gamma * (Of > 0)

    max_value = np.max(I_gamma)

    if max_value > 0:
        I_rgb = I_rgb / max_value


    if mode == "simulation":

        fig, axs = plt.subplots(2, 3, figsize=(12, 8))

        extent = [
            -size / 2,
            size / 2,
            -size / 2,
            size / 2
        ]

        axs[0, 0].imshow(Of, cmap="gray", extent=extent, origin="lower")
        axs[0, 0].set_title("Front Grid")

        axs[0, 1].imshow(Or, cmap="gray", extent=extent, origin="lower")
        axs[0, 1].set_title("Rear Grid")

        axs[0, 2].imshow(Or_shift, cmap="gray", extent=extent, origin="lower")
        axs[0, 2].set_title("Shifted Rear Grid")

        axs[1, 0].imshow(I_ideal, cmap="gray", extent=extent, origin="lower")
        axs[1, 0].set_title("Ideal Image")

        axs[1, 1].imshow(I_gamma, cmap="gray", extent=extent, origin="lower")
        axs[1, 1].set_title(
            f"Simulated Image\n"
            f"sigma={sigma}, noise={noise_switch}"
        )

        axs[1, 2].imshow(I_rgb, extent=extent, origin="lower")
        axs[1, 2].set_title("RGB Overlay")

        for ax in axs.flat:
            ax.set_xlabel("x [mm]")
            ax.set_ylabel("y [mm]")

        plt.tight_layout()
        plt.show()

        print("sv:", sv)
        print("delta:", delta_x, delta_y)
        print("I_gamma_max:", max_value)

        return {
            "intensity": I_gamma,
            "I_ideal": I_ideal,
            "I_blur": I_blur,
            "I_final": I_final,
            "Of": Of,
            "Or": Or,
            "Or_shift": Or_shift,
            "Vr": Vr,
            "sv": sv,
            "delta_x": delta_x,
            "delta_y": delta_y,
            "image": I_rgb,
        }

    elif mode == "inverse":

        return {
            "intensity": I_gamma,
            "I_ideal": I_ideal,
            "I_blur": I_blur,
            "I_final": I_final,
            "Of": Of,
            "Or": Or,
            "Or_shift": Or_shift,
            "Vr": Vr,
            "sv": sv,
            "delta_x": delta_x,
            "delta_y": delta_y,
            "image": I_rgb,
        }

In [ ]:
def parallel_grid_search(d_test, dx_range, dy_range, I_refs, thetas, phis, N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0,
                         L0 = 1.0, rho_f = 1.0, rho_r = 1.0, Ibg = 0.1, distance_func_str = "MSE", combined_error = "mean", sigma = 0,
                         noise_switch = False, poisson_scale = 1000, gaussian_std = 0.01, gamma_switch = False, gamma = 0.8):

    """
    Performs a grid search over dx and dy for a fixed value of d.

    For each tested parameter combination, one simulated candidate image
    is generated for each provided camera view and compared with the
    corresponding reference image.

    Parameters
    ----------
    d_test : float
        Tested distance between the front and rear grids in mm.

    dx_range : iterable
        Values of the lateral displacement dx to be tested in mm.

    dy_range : iterable
        Values of the lateral displacement dy to be tested in mm.

    I_refs : list of ndarray
        Reference intensity images.
        One or two reference images are supported.

    thetas : iterable
        Polar viewing angles corresponding to the reference images
        in radians.

    phis : iterable
        Azimuth viewing angles corresponding to the reference images
        in radians.

    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the
        front and rear grids.

    Ibg : float
        Constant background intensity.

    distance_func_str : str
        Distance metric used to compare reference and candidate images.

    combined_error : {"mean", "max"}
        Method used to combine the errors of two camera views.
        Ignored when only one reference image is provided.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, noise is also applied to the simulated candidate images.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied to the simulated
        candidate images.

    gamma : float
        Gamma exponent used for gamma correction.

    Returns
    -------
    local_results : list of dict
        Results for all tested combinations of dx and dy.

        Each dictionary contains:

        "error" : float
            Combined image error.

        "dm1" : float
            Error of the first camera view.

        "dm2" : float
            Error of the second camera view.
            Set to -1 if only one reference image is used.

        "d" : float
            Tested grid distance.

        "dx" : float
            Tested lateral displacement in x-direction.

        "dy" : float
            Tested lateral displacement in y-direction.
    """

    local_results = []

    for dx_test in dx_range:
        for dy_test in dy_range:

            errors = []

            for I_ref, theta, phi in zip(I_refs, thetas, phis):

                I_res = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                d = d_test, dx = dx_test, dy = dy_test, theta = theta, phi = phi, sigma = sigma, noise_switch = noise_switch,
                poisson_scale = poisson_scale, gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")
                
                I_test = I_res["intensity"]
                
                dm = distance_function(I_ref, I_test, distance_func_str)
                errors.append(dm)

            if(len(I_refs) == 1):
                dm1 = errors[0]
                dm2 = -1
                err = dm1

            else:
                dm1 = errors[0]
                dm2 = errors[1]
                
                if combined_error == "mean":
                    err = (dm1 + dm2) / 2
                
                elif combined_error == "max":
                    err = max(dm1, dm2)
            
                else:
                    raise ValueError(
                        "combined_error muss 'mean' oder 'max' sein."
                    )
                
            local_results.append({
                "error": err,
                "dm1": dm1,
                "dm2": dm2,
                "d": d_test,
                "dx": dx_test,
                "dy": dy_test
            })

    return local_results

In [ ]:
def grid_search_topk(d_range, dx_range, dy_range, I_refs, thetas, phis, N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0, K = 5,
                     L0 = 1.0, rho_f = 1.0, rho_r = 1.0, Ibg = 0.1, n_jobs = 5, distance_func_str = "MSE",
                     combined_error = "mean", sigma = 0, noise_switch = False, poisson_scale = 1000,
                     gaussian_std = 0.01, gamma_switch = False, gamma = 0.8):

    """
    Performs a parallel grid search over d, dx and dy and returns
    the best parameter combinations.

    For each tested value of d, a grid search over dx and dy is
    evaluated in parallel using parallel_grid_search(). One or two
    camera views can be used.

    Parameters
    ----------
    d_range : iterable
        Values of the grid distance d to be tested in mm.

    dx_range : iterable
        Values of the lateral displacement dx to be tested in mm.

    dy_range : iterable
        Values of the lateral displacement dy to be tested in mm.

    I_refs : list of ndarray
        Reference intensity images.
        One or two reference images are supported.

    thetas : iterable
        Polar viewing angles corresponding to the reference images
        in radians.

    phis : iterable
        Azimuth viewing angles corresponding to the reference images
        in radians.

    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    K : int
        Number of best parameter combinations returned.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the
        front and rear grids.

    Ibg : float
        Constant background intensity.

    n_jobs : int
        Number of parallel worker processes used for the grid search.

    distance_func_str : str
        Distance metric used to compare reference and candidate images.

    combined_error : {"mean", "max"}
        Method used to combine the errors of two camera views.
        Ignored when only one reference image is provided.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, noise is applied to the simulated candidate images.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied to the simulated
        candidate images.

    gamma : float
        Gamma exponent used for gamma correction.

    Returns
    -------
    top_k : list of dict
        The K parameter combinations with the smallest combined error.

    sorted_results : list of dict
        All grid-search results sorted in ascending order by error.

    results : list of dict
        All grid-search results in their original collection order.
    """

    if not (len(I_refs) == len(thetas) == len(phis)):
        raise ValueError("I_refs, thetas and phis must have the same length.")

    if len(I_refs) not in (1, 2):
        raise ValueError("One or two camera views are supported.")

    jobs = []

    for d_test in d_range:
        jobs.append(
            delayed(parallel_grid_search)(
                d_test,
                dx_range,
                dy_range,
                I_refs,
                thetas,
                phis,
                N,
                size,
                p,
                Df,
                Dr,
                L0,
                rho_f,
                rho_r,
                Ibg,
                distance_func_str,
                combined_error,
                sigma,
                noise_switch,
                poisson_scale,
                gaussian_std,
                gamma_switch,
                gamma
            )
        )

    parallel_results = Parallel(n_jobs = n_jobs)(jobs)

    results = []
    for res in parallel_results:
        results.extend(res)

    sorted_results = sorted(results.copy(), key = lambda x: x["error"])
    
    return sorted_results[:K], sorted_results, results

In [ ]:
def distance_function(I1, I2, function = "MSE"):

    """
    Computes the distance between two intensity images.

    Parameters
    ----------
    I1, I2 : ndarray
        Intensity images to be compared. Both arrays should have
        the same shape.

    function : {"MSE", "MAE", "NMSE"}
        Distance metric used for comparison.

        "MSE"
            Mean squared error.

        "MAE"
            Mean absolute error.

        "NMSE"
            Normalized mean squared error, normalized by the mean
            squared intensity of I1.

    Returns
    -------
    distance : float
        Computed distance between the two images.

    Raises
    ------
    ValueError
        If an unsupported distance metric is specified.
    """

    if function == "MSE":
        return np.mean((I1 - I2)**2)
    
    elif function == "MAE":
        return np.mean(np.abs(I1 - I2))
    
    elif function == "NMSE":
        mse = np.mean((I1 - I2)**2)
        norm = np.mean(I1**2)
        return mse / norm
    
    else:
        raise ValueError(
            f"Unknown distance function: {function}. Use 'MSE', 'MAE' oder 'NMSE'."
        )

In [ ]:
def save_results(all_results, filename = "grid_search_results.csv", with_theta = False, with_phi = False):

    """
    Saves grid-search results to a CSV file and returns them
    as a pandas DataFrame.

    Parameters
    ----------
    all_results : list
        Grid-search results to save.

        If with_theta=False and with_phi=False, the expected
        structure is a list of result lists, where each inner list
        belongs to one reference case.

        If with_theta=True or with_phi=True, the expected structure
        is a flat list of dictionaries containing the keys
        "ref_id" and "result" as well as "theta_study" or
        "phi_study", respectively.

    filename : str
        Name or path of the output CSV file.

    with_theta : bool
        If True, theta study values are included in the output
        DataFrame and CSV file.

    with_phi : bool
        If True, phi study values are included in the output
        DataFrame and CSV file.

    Returns
    -------
    df : pandas.DataFrame
        DataFrame containing the grid-search results with the
        columns ref_id, d, dx, dy, error, dm1 and dm2.
        Depending on the selected study mode, theta_study or
        phi_study is included as an additional column.
    """

    history = []

    if with_theta or with_phi:
        for entry in all_results:
            result = entry["result"]

            data = {
                "ref_id": entry["ref_id"],
                "d": result["d"],
                "dx": result["dx"],
                "dy": result["dy"],
                "error": result["error"],
                "dm1": result["dm1"],
                "dm2": result.get("dm2", -1),
            }

            if with_theta:
                data["theta_study"] = entry["theta_study"]

            if with_phi:
                data["phi_study"] = entry["phi_study"]

            history.append(data)

    else:
        for ref_id, results in enumerate(all_results, start = 1):
            for result in results:
                history.append({
                    "ref_id": ref_id,
                    "d": result["d"],
                    "dx": result["dx"],
                    "dy": result["dy"],
                    "error": result["error"],
                    "dm1": result["dm1"],
                    "dm2": result.get("dm2", -1),
                })

    df = pd.DataFrame(history)
    df.to_csv(filename, index = False)

    return df

In [ ]:
def periodic_width(values, p):

    """
    Computes the minimal width containing all values on a periodic interval.

    Parameters
    ----------
    values : array-like
        Values on a periodic interval of length p.

    p : float
        Period length.

    Returns
    -------
    width : float
        Minimal periodic interval width containing all unique values.
        Returns 0 if zero or one unique value is provided.
    """

    values = np.sort(np.unique(values))

    if len(values) <= 1:
        return 0.0

    gaps = np.diff(values)
    period_gap = p - values[-1] + values[0]
    gaps = np.append(gaps, period_gap)

    return p - np.max(gaps)

In [ ]:
def param_analysis_1d(ref_param, variable, values, N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0, L0 = 1.0, rho_f = 1.0, rho_r = 1.0, 
                      Ibg = 0.1, distance_func_str = "MSE", theta_study = None, phi_study = None, K = 5, sigma = 0, noise_switch = False,
                      poisson_scale = 1000, gaussian_std = 0.01, gamma_switch = False, gamma = 0.8, run_id = None, save_csv = True,
                      filename_prefix = "1d_sens_analysis", filename_mode = "full", plot = True):

    """
    Performs a one-dimensional sensitivity analysis for one of the
    geometric parameters d, dx or dy.

    The selected parameter is varied over the specified values while
    the remaining geometric parameters are kept at their reference
    values. Optionally, the analysis can be repeated for different
    polar or azimuth viewing angles.

    Parameters
    ----------
    ref_param : list of dict
        Reference cases containing the ground-truth parameters d, dx,
        dy, theta and phi. A label may additionally be provided for
        plotting.

    variable : {"d", "dx", "dy"}
        Geometric parameter to be varied.

    values : array-like
        Values of the selected parameter to be tested in mm.

    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the front
        and rear grids.

    Ibg : float
        Constant background intensity.

    distance_func_str : str
        Distance metric used for image comparison, e.g. "MSE",
        "MAE" or "NMSE".

    theta_study : array-like or None
        Optional polar viewing angles in radians. If provided, a
        separate one-dimensional sensitivity analysis is performed
        for each specified theta value while phi remains fixed at
        the corresponding reference value.

    phi_study : array-like or None
        Optional azimuth viewing angles in radians. If provided, a
        separate one-dimensional sensitivity analysis is performed
        for each specified phi value while theta remains fixed at
        the corresponding reference value.

    K : int
        Number of best candidates requested internally from
        grid_search_topk. The returned sensitivity DataFrame still
        contains all evaluated parameter combinations.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, noise is applied to the generated reference images.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied.

    gamma : float
        Gamma exponent used for gamma correction.

    run_id : int, str or None
        Optional identifier included in the output filename.

    save_csv : bool
        If True, the results are saved as a CSV file.

    filename_prefix : str
        Prefix used for the output CSV filename.

    filename_mode : {"full", "short"}
        Controls the amount of configuration information included
        in the output filename.

    plot : bool
        If True, the resulting one-dimensional sensitivity curves
        are plotted.

    Returns
    -------
    df : pandas.DataFrame
        Sensitivity-analysis results. The DataFrame contains the
        reference ID, tested geometric parameters d, dx and dy,
        the total image error and the individual view errors dm1
        and dm2.

        If theta_study is used, an additional column
        "theta_study" contains the investigated angle in degrees.

        If phi_study is used, an additional column
        "phi_study" contains the investigated angle in degrees.

    Raises
    ------
    ValueError
        If variable is not "d", "dx" or "dy", if theta_study and
        phi_study are specified simultaneously, or if filename_mode
        is neither "full" nor "short".
    """

    if variable not in ("d", "dx", "dy"):
        raise ValueError("variable must be 'd', 'dx' or 'dy'")

    if theta_study is not None and phi_study is not None:
        raise ValueError("Use either theta_study or phi_study, not both.")

    if filename_mode not in ("full", "short"):
        raise ValueError("filename_mode must be 'full' or 'short'")

    if theta_study is not None:
        theta_string = "Study"

    else:
        theta_string = "-".join(str(np.rad2deg(ref['theta'])) for ref in ref_param)
        

    if phi_study is not None:
        phi_string = "Study"

    else:
        phi_string = "-".join(str(np.rad2deg(ref['phi'])) for ref in ref_param)


    if filename_mode == "full":
        filename = (
            f"{filename_prefix}_"
            f"var_{variable}_"
            f"run_{run_id}_"
            f"N_{N}_"
            f"size_{size}_"
            f"p_{p}_"
            f"Df_{Df}_"
            f"Dr_{Dr}_"
            # f"L0_{L0}_"
            # f"rho_f_{rho_f}_"
            # f"rho_r_{rho_r}_"
            # f"Ibg_{Ibg}_"
            f"theta_{theta_string}_"
            f"phi_{phi_string}_"
            f"metric_{distance_func_str}_"
            f"sigma_{sigma}_"
            f"noise_{noise_switch}_"
            f"poisson_{poisson_scale}_"
            f"gaussian_{gaussian_std}_"
            f"gamma_on_{gamma_switch}_"
            f"gamma_{gamma}.csv"
        )

    else:
        filename = (
            f"{filename_prefix}_"
            f"var_{variable}_"
            f"run_{run_id}_"
            f"theta_{theta_string}_"
            f"phi_{phi_string}_"
            f"metric_{distance_func_str}.csv"
        )

    if theta_study is None and phi_study is None:
        all_results = []

        for ref in ref_param:
            ref_sim = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                 d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = ref["theta"], phi = ref["phi"],
                                 sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale,
                                 gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")

            if variable == "d":
                d_values = values
                dx_values = [ref["dx"]]
                dy_values = [ref["dy"]]

            elif variable == "dx":
                d_values = [ref["d"]]
                dx_values = values
                dy_values = [ref["dy"]]

            elif variable == "dy":
                d_values = [ref["d"]]
                dx_values = [ref["dx"]]
                dy_values = values

            _, _, results = grid_search_topk(d_values, dx_values, dy_values, [ref_sim["intensity"]],
                                    N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                    thetas = [ref["theta"]], phis = [ref["phi"]], K = K, distance_func_str = distance_func_str,
                                    gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)

            all_results.append(results)

        if save_csv:
            df = save_results(all_results, filename = filename)

        else:
            history = []

            for ref_id, results in enumerate(all_results, start = 1):
                for result in results:
                    history.append({
                        "ref_id": ref_id,
                        "d": result["d"],
                        "dx": result["dx"],
                        "dy": result["dy"],
                        "error": result["error"],
                        "dm1": result["dm1"],
                        "dm2": -1,
                    })

            df = pd.DataFrame(history)

    elif theta_study is not None:
        all_results = []

        for angle in theta_study:
            for ref_id, ref in enumerate(ref_param, start = 1):
                ref_angle = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                       d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = angle, phi = ref["phi"],
                                       sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale, gaussian_std = gaussian_std,
                                       gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")
                
                if variable == "d":
                    d_values = values
                    dx_values = [ref["dx"]]
                    dy_values = [ref["dy"]]

                elif variable == "dx":
                    d_values = [ref["d"]]
                    dx_values = values
                    dy_values = [ref["dy"]]

                elif variable == "dy":
                    d_values = [ref["d"]]
                    dx_values = [ref["dx"]]
                    dy_values = values

                _, _, results = grid_search_topk(d_values, dx_values, dy_values, [ref_angle["intensity"]],
                                N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                thetas = [angle], phis = [ref["phi"]], K = K, distance_func_str = distance_func_str,
                                gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)

                for result in results:
                    all_results.append({
                        "ref_id": ref_id,
                        "theta_study": np.rad2deg(angle),
                        "result": result
                    })

        if save_csv:
            df = save_results(all_results, filename = filename, with_theta = True)

        else:
            history = []
            for entry in all_results:
                result = entry["result"]

                history.append({
                    "ref_id": entry["ref_id"],
                    "theta_study": entry["theta_study"],
                    "d": result["d"],
                    "dx": result["dx"],
                    "dy": result["dy"],
                    "error": result["error"],
                    "dm1": result["dm1"],
                    "dm2": -1,
                })

            df = pd.DataFrame(history)

    elif phi_study is not None:
        all_results = []

        for angle in phi_study:
            for ref_id, ref in enumerate(ref_param, start = 1):
                ref_angle = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                        d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = ref["theta"], phi = angle,
                                        sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale,
                                        gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")


                if variable == "d":
                    d_values = values
                    dx_values = [ref["dx"]]
                    dy_values = [ref["dy"]]

                elif variable == "dx":
                    d_values = [ref["d"]]
                    dx_values = values
                    dy_values = [ref["dy"]]

                elif variable == "dy":
                    d_values = [ref["d"]]
                    dx_values = [ref["dx"]]
                    dy_values = values

                _, _, results = grid_search_topk(d_values, dx_values, dy_values, [ref_angle["intensity"]],
                                                 N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                                 thetas = [ref["theta"]], phis = [angle], K = K, distance_func_str = distance_func_str,
                                                 gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)

                for result in results:
                    all_results.append({
                        "ref_id": ref_id,
                        "phi_study": np.rad2deg(angle),
                        "result": result
                    })

        if save_csv:
            df = save_results(all_results, filename = filename, with_phi = True)

        else:
            history = []

            for entry in all_results:
                result = entry["result"]
                history.append({
                    "ref_id": entry["ref_id"],
                    "phi_study": entry["phi_study"],
                    "d": result["d"],
                    "dx": result["dx"],
                    "dy": result["dy"],
                    "error": result["error"],
                    "dm1": result["dm1"],
                    "dm2": -1,
                })

            df = pd.DataFrame(history)

    if plot:
        plot_sensitivity_1d(df = df, ref_param = ref_param, variable = variable, distance_func_str = distance_func_str,
                            theta_study = theta_study, phi_study = phi_study)

    return df

In [ ]:
def plot_sensitivity_1d(
    df,
    ref_param,
    variable,
    distance_func_str,
    theta_study=None,
    phi_study=None
):
    """
    Plots the results of a one-dimensional sensitivity analysis.

    The function visualizes the image error as a function of one
    geometric parameter d, dx or dy. If theta_study or phi_study
    is provided, separate curves are plotted for the investigated
    viewing angles.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the sensitivity-analysis results.
        Expected columns include "ref_id", "d", "dx", "dy",
        "error" and, depending on the study mode, "theta_study"
        or "phi_study".

    ref_param : list of dict
        Reference cases containing the ground-truth parameter
        values and a label for plotting.

    variable : {"d", "dx", "dy"}
        Geometric parameter shown on the x-axis.

    distance_func_str : str
        Name of the distance metric shown on the y-axis,
        e.g. "MSE", "MAE" or "NMSE".

    theta_study : array-like or None
        Indicates that a theta-angle study is plotted. The
        corresponding angle values are expected in the
        "theta_study" column of df and are displayed in degrees.

    phi_study : array-like or None
        Indicates that a phi-angle study is plotted. The
        corresponding angle values are expected in the
        "phi_study" column of df and are displayed in degrees.

    Returns
    -------
    None
        The function displays matplotlib figures and does not
        return a value.
    """

    if variable == "dx":
        variable_label = r"$\Delta x$ [mm]"
    elif variable == "dy":
        variable_label = r"$\Delta y$ [mm]"
    elif variable == "d":
        variable_label = r"$d$ [mm]"
    else:
        variable_label = variable

    if theta_study is not None or phi_study is not None:

        for ref_id in df["ref_id"].unique():

            ref = ref_param[ref_id - 1]

            fig, ax = plt.subplots(figsize=(8, 6))

            if theta_study is not None:
                study_column = "theta_study"
                angle_symbol = r"$\theta$"
            else:
                study_column = "phi_study"
                angle_symbol = r"$\phi$"

            for angle_deg in sorted(df[study_column].unique()):

                df_ref = df[
                    (df["ref_id"] == ref_id)
                    & np.isclose(df[study_column], angle_deg)
                ].sort_values(variable)

                x = df_ref[variable].to_numpy()
                errors = df_ref["error"].to_numpy()

                if len(errors) == 0:
                    continue

                i_min = np.argmin(errors)

                print("-" * 50)
                print(
                    f"{ref['label']} | "
                    f"{angle_symbol} = {angle_deg:.0f}°"
                )
                print("-" * 50)

                print(
                    f"Minimal {distance_func_str} = "
                    f"{errors[i_min]}"
                )

                print(
                    f"Best {variable} = "
                    f"{x[i_min]}"
                )

                ax.plot(
                    x,
                    errors,
                    label=fr"{angle_symbol} = {angle_deg:.0f}°"
                )

            ax.axvline(
                ref[variable],
                linestyle="--",
                linewidth=1.5,
                label="Ground Truth"
            )

            ax.set_xlabel(variable_label)
            ax.set_ylabel(distance_func_str)

            ax.set_title(
                f"1D-Sensitivity Analysis: "
                f"{variable} – {ref['label']}"
            )

            ax.grid()

            ax.legend(
                loc="upper center",
                bbox_to_anchor=(0.5, -0.15),
                ncol=3
            )

            fig.tight_layout()
            plt.show()

    else:

        all_errors = []

        fig, ax = plt.subplots(figsize=(8, 6))

        for ref_id in df["ref_id"].unique():

            ref = ref_param[ref_id - 1]

            df_ref = (
                df[df["ref_id"] == ref_id]
                .sort_values(variable)
            )

            x = df_ref[variable].to_numpy()
            errors = df_ref["error"].to_numpy()

            all_errors.append(errors)

            ax.plot(
                x,
                errors,
                label=ref["label"]
            )

        if len(all_errors) > 0:
            lengths = [len(e) for e in all_errors]

            if len(set(lengths)) == 1:
                error_mean = np.mean(
                    all_errors,
                    axis=0
                )

                # ax.plot(
                #     x,
                #     error_mean,
                #     label="Mean",
                #     linewidth=2
                # )

        ax.set_xlabel(variable_label)
        ax.set_ylabel(distance_func_str)

        ax.set_title(
            f"Sensitivity of {variable}"
        )

        ax.grid()

        ax.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.15),
            ncol=2
        )

        fig.tight_layout()
        plt.show()

In [ ]:
def param_analysis_2d(ref_param, variable_x, values_x, variable_y, values_y, N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0, L0 = 1.0, rho_f = 1.0,
                      rho_r = 1.0, Ibg = 0.1, distance_func_str = "MSE", theta_study = None, phi_study = None, K = 5, sigma = 0, noise_switch = False,
                      poisson_scale = 1000, gaussian_std = 0.01, gamma_switch = False, gamma = 0.8, run_id = None, threshold_abs = 0.01,
                      save_csv = True, filename_prefix = "2d_sens_analysis", filename_mode = "full", plot = True):

    """
    Performs a two-dimensional sensitivity analysis for two of the
    geometric parameters d, dx and dy.

    The two selected parameters are varied over the specified value
    ranges while the remaining geometric parameter is kept at its
    reference value. Optionally, the analysis can be repeated for
    different polar or azimuth viewing angles.

    Parameters
    ----------
    ref_param : list of dict
        Reference cases containing the ground-truth parameters d, dx,
        dy, theta and phi. A label may additionally be provided for
        plotting.

    variable_x : {"d", "dx", "dy"}
        Geometric parameter varied along the x-axis.

    values_x : array-like
        Values of variable_x to be tested in mm.

    variable_y : {"d", "dx", "dy"}
        Geometric parameter varied along the y-axis.

    values_y : array-like
        Values of variable_y to be tested in mm.

    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the front
        and rear grids.

    Ibg : float
        Constant background intensity.

    distance_func_str : str
        Distance metric used for image comparison, e.g. "MSE",
        "MAE" or "NMSE".

    theta_study : array-like or None
        Optional polar viewing angles in radians. If provided, a
        separate two-dimensional sensitivity analysis is performed
        for each specified theta value while phi remains fixed at
        the corresponding reference value.

    phi_study : array-like or None
        Optional azimuth viewing angles in radians. If provided, a
        separate two-dimensional sensitivity analysis is performed
        for each specified phi value while theta remains fixed at
        the corresponding reference value.

    K : int
        Number of best candidates requested internally from
        grid_search_topk. The returned DataFrame contains all
        evaluated parameter combinations.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, noise is applied to the generated reference images.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied.

    gamma : float
        Gamma exponent used for gamma correction.

    run_id : int, str or None
        Optional identifier included in the output filename.

    threshold_abs : float
        Absolute error threshold used for identifying approximately
        equivalent solutions in the sensitivity plot.

    save_csv : bool
        If True, the results are saved as a CSV file.

    filename_prefix : str
        Prefix used for the output CSV filename.

    filename_mode : {"full", "short"}
        Controls the amount of configuration information included
        in the output filename.

    plot : bool
        If True, the two-dimensional sensitivity results are plotted.

    Returns
    -------
    df : pandas.DataFrame
        Results of the two-dimensional sensitivity analysis.

        The DataFrame contains the reference ID, tested geometric
        parameters d, dx and dy, the total image error and the
        individual view errors dm1 and dm2.

        If theta_study is used, an additional column "theta_study"
        contains the investigated angle in degrees.

        If phi_study is used, an additional column "phi_study"
        contains the investigated angle in degrees.

    Raises
    ------
    ValueError
        If variable_x or variable_y is not "d", "dx" or "dy", if both
        variables are identical, if theta_study and phi_study are
        specified simultaneously, or if filename_mode is neither
        "full" nor "short".
    """

    if variable_x not in ("d", "dx", "dy"):
        raise ValueError("variable_x must be 'd', 'dx' or 'dy'")

    if variable_y not in ("d", "dx", "dy"):
        raise ValueError("variable_y must be 'd', 'dx' or 'dy'")

    if variable_x == variable_y:
        raise ValueError("variable_x and variable_y must be different.")

    if theta_study is not None and phi_study is not None:
        raise ValueError("Use either theta_study or phi_study, not both.")

    if filename_mode not in ("full", "short"):
        raise ValueError("filename_mode must be 'full' or 'short'")


    if theta_study is not None:
        theta_string = "Study"
    else:
        theta_string = "-".join(str(np.rad2deg(ref["theta"])) for ref in ref_param)

    if phi_study is not None:
        phi_string = "Study"
    else:
        phi_string = "-".join(str(np.rad2deg(ref["phi"])) for ref in ref_param)

    if filename_mode == "full":
        filename = (
            f"{filename_prefix}_"
            f"var_{variable_x}_{variable_y}_"
            f"run_{run_id}_"
            f"N_{N}_"
            f"size_{size}_"
            f"p_{p}_"
            f"Df_{Df}_"
            f"Dr_{Dr}_"
            # f"L0_{L0}_"
            # f"rho_f_{rho_f}_"
            # f"rho_r_{rho_r}_"
            # f"Ibg_{Ibg}_"
            f"theta_{theta_string}_"
            f"phi_{phi_string}_"
            f"metric_{distance_func_str}_"
            f"sigma_{sigma}_"
            f"noise_{noise_switch}_"
            f"poisson_{poisson_scale}_"
            f"gaussian_{gaussian_std}_"
            f"gamma_on_{gamma_switch}_"
            f"gamma_{gamma}.csv"
        )

    else:
        filename = (
            f"{filename_prefix}_"
            f"var_{variable_x}_{variable_y}_"
            f"run_{run_id}_"
            f"theta_{theta_string}_"
            f"phi_{phi_string}_"
            f"metric_{distance_func_str}.csv"
        )

    if theta_study is None and phi_study is None:
        all_results = []

        for ref in ref_param:
            ref_sim = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                 d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = ref["theta"], phi = ref["phi"],
                                 sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale,
                                 gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")

            values = {
                "d": [ref["d"]],
                "dx": [ref["dx"]],
                "dy": [ref["dy"]]
            }
            
            values[variable_x] = values_x
            values[variable_y] = values_y

            d_values = values["d"]
            dx_values = values["dx"]
            dy_values = values["dy"]

            _, _, results = grid_search_topk(d_values, dx_values, dy_values, [ref_sim["intensity"]],
                                    N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                    thetas = [ref["theta"]], phis = [ref["phi"]], K = K, distance_func_str = distance_func_str,
                                    gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)

            all_results.append(results)

        if save_csv:
            df = save_results(all_results, filename = filename)

        else:
            history = []

            for ref_id, results in enumerate(all_results, start = 1):
                for result in results:
                    history.append({
                        "ref_id": ref_id,
                        "d": result["d"],
                        "dx": result["dx"],
                        "dy": result["dy"],
                        "error": result["error"],
                        "dm1": result["dm1"],
                        "dm2": -1
                    })

            df = pd.DataFrame(history)

    elif theta_study is not None:
        all_results = []

        for angle in theta_study:
            for ref_id, ref in enumerate(ref_param, start = 1):
                ref_angle = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                       d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = angle, phi = ref["phi"],
                                       sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale,
                                       gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")

                values = {"d": [ref["d"]],
                          "dx": [ref["dx"]],
                          "dy": [ref["dy"]]
                          }
                
                values[variable_x] = values_x
                values[variable_y] = values_y

                d_values = values["d"]
                dx_values = values["dx"]
                dy_values = values["dy"]

                _, _, results = grid_search_topk(d_values, dx_values, dy_values, [ref_angle["intensity"]],
                                                 N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                                 thetas = [angle], phis = [ref["phi"]], K = K, distance_func_str = distance_func_str,
                                                 gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)

                for result in results:

                    all_results.append({
                        "ref_id": ref_id,
                        "theta_study": np.rad2deg(angle),
                        "result": result
                    })

        if save_csv:
            df = save_results(all_results, filename = filename, with_theta = True)

        else:
            history = []

            for entry in all_results:
                result = entry["result"]

                history.append({
                    "ref_id": entry["ref_id"],
                    "theta_study": entry["theta_study"],
                    "d": result["d"],
                    "dx": result["dx"],
                    "dy": result["dy"],
                    "error": result["error"],
                    "dm1": result["dm1"],
                    "dm2": -1
                })

            df = pd.DataFrame(history)


    elif phi_study is not None:
        all_results = []

        for angle in phi_study:
            for ref_id, ref in enumerate(ref_param, start = 1):
                ref_angle = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                       d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = ref["theta"], phi = angle,
                                       sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale,
                                       gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")

                values = {"d": [ref["d"]],
                          "dx": [ref["dx"]],
                          "dy": [ref["dy"]]
                          }
                
                values[variable_x] = values_x
                values[variable_y] = values_y

                d_values = values["d"]
                dx_values = values["dx"]
                dy_values = values["dy"]

                _, _, results = grid_search_topk(d_values, dx_values, dy_values, [ref_angle["intensity"]],
                                                 N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                                 thetas = [ref["theta"]], phis = [angle], K = K, distance_func_str = distance_func_str,
                                                 gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)


                for result in results:
                    all_results.append({
                        "ref_id": ref_id,
                        "phi_study": np.rad2deg(angle),
                        "result": result
                    })

        if save_csv:
            df = save_results(all_results, filename = filename, with_phi = True)

        else:
            history = []

            for entry in all_results:
                result = entry["result"]

                history.append({
                    "ref_id": entry["ref_id"],
                    "phi_study": entry["phi_study"],
                    "d": result["d"],
                    "dx": result["dx"],
                    "dy": result["dy"],
                    "error": result["error"],
                    "dm1": result["dm1"],
                    "dm2": -1
                })

            df = pd.DataFrame(history)

    if plot:
        plot_sensitivity_2d(df = df, ref_param = ref_param, variable_x = variable_x, values_x = values_x,
                            variable_y = variable_y, values_y = values_y, p = p,
                            distance_func_str = distance_func_str, threshold_abs = threshold_abs,
                            theta_study = theta_study, phi_study = phi_study)

    return df

In [ ]:
def plot_sensitivity_2d(
    df,
    ref_param,
    variable_x,
    values_x,
    variable_y,
    values_y,
    p,
    distance_func_str="MSE",
    threshold_abs=0.01,
    theta_study=None,
    phi_study=None,
    theta_views=None,
    phi_views=None
):
    """
    Plots the results of a two-dimensional sensitivity analysis.

    The function visualizes the image error over two geometric parameters
    using a heatmap. It additionally identifies the global minimum,
    approximately equivalent solutions below an absolute error threshold,
    the spatial extent of the equivalent region, and the error at the
    ground-truth parameter combination.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing the results of the two-dimensional
        sensitivity analysis. Expected columns include "ref_id",
        "d", "dx", "dy", "error" and, depending on the study mode,
        "theta_study" or "phi_study".

    ref_param : list of dict
        Reference cases containing the ground-truth parameters d, dx,
        dy and a label for plotting.

    variable_x : {"d", "dx", "dy"}
        Geometric parameter shown on the x-axis.

    values_x : array-like
        Tested values of variable_x in mm.

    variable_y : {"d", "dx", "dy"}
        Geometric parameter shown on the y-axis.

    values_y : array-like
        Tested values of variable_y in mm.

    p : float
        Grid period in mm. Used to determine the periodic width of
        equivalent solution regions for dx and dy.

    distance_func_str : str
        Name of the distance metric shown in the colorbar and printed
        output, e.g. "MSE", "MAE" or "NMSE".

    threshold_abs : float
        Absolute error threshold used to classify parameter combinations
        as approximately equivalent solutions.

    theta_study : array-like or None
        Indicates that results from a theta-angle study are plotted.
        The corresponding angle values are read from the "theta_study"
        column of df and are interpreted in degrees.

    phi_study : array-like or None
        Indicates that results from a phi-angle study are plotted.
        The corresponding angle values are read from the "phi_study"
        column of df and are interpreted in degrees.

    theta_views : iterable or None
        Polar viewing angles in radians used for a two-view analysis.
        If two values are provided together with phi_views, they are
        included in the plot title.

    phi_views : iterable or None
        Azimuth viewing angles in radians used for a two-view analysis.
        If two values are provided together with theta_views, they are
        included in the plot title.

    Returns
    -------
    None
        The function prints diagnostic information and displays
        matplotlib figures but does not return a value.
    """
    # Achsenbeschriftungen
    variable_labels = {
        "dx": r"$\Delta x$ [mm]",
        "dy": r"$\Delta y$ [mm]",
        "d": r"$d$ [mm]"
    }

    x_label = variable_labels.get(variable_x, variable_x)
    y_label = variable_labels.get(variable_y, variable_y)

    # Schriftgrößen
    label_fontsize = 17
    tick_fontsize = 13
    title_fontsize = 17
    legend_fontsize = 15
    colorbar_fontsize = 16

    if theta_study is not None:
        study_column = "theta_study"
        angle_symbol = r"$\theta$"
        angle_values = sorted(df[study_column].unique())

    elif phi_study is not None:
        study_column = "phi_study"
        angle_symbol = r"$\phi$"
        angle_values = sorted(df[study_column].unique())

    else:
        study_column = None
        angle_symbol = None
        angle_values = [None]

    for angle_deg in angle_values:

        for ref_id in df["ref_id"].unique():

            ref = ref_param[ref_id - 1]

            if study_column is None:
                df_ref = df[df["ref_id"] == ref_id]
            else:
                df_ref = df[
                    (df["ref_id"] == ref_id)
                    & np.isclose(df[study_column], angle_deg)
                ]

            err_matrix = np.full(
                (len(values_y), len(values_x)),
                np.nan
            )

            for _, row in df_ref.iterrows():

                i = np.argmin(
                    np.abs(values_y - row[variable_y])
                )

                j = np.argmin(
                    np.abs(values_x - row[variable_x])
                )

                err_matrix[i, j] = row["error"]

            D_min = np.nanmin(err_matrix)

            global_minima = np.isclose(
                err_matrix,
                D_min,
                rtol=0,
                atol=1e-12
            )

            N_global_minima = np.sum(global_minima)

            min_idx = np.unravel_index(
                np.nanargmin(err_matrix),
                err_matrix.shape
            )

            y_best = values_y[min_idx[0]]
            x_best = values_x[min_idx[1]]

            equivalent = (
                err_matrix <= threshold_abs
            )

            N_equivalent = np.sum(equivalent)

            N_total = np.sum(
                ~np.isnan(err_matrix)
            )

            percentage = (
                100
                * N_equivalent
                / N_total
            )

            rows, cols = np.where(equivalent)

            if len(rows) > 0:

                y_region = values_y[rows]
                x_region = values_x[cols]

                if variable_x in ("dx", "dy"):
                    x_width = periodic_width(
                        x_region,
                        p
                    )
                else:
                    x_width = (
                        x_region.max()
                        - x_region.min()
                    )

                if variable_y in ("dx", "dy"):
                    y_width = periodic_width(
                        y_region,
                        p
                    )
                else:
                    y_width = (
                        y_region.max()
                        - y_region.min()
                    )

            else:
                x_width = 0
                y_width = 0

            i_gt = np.argmin(
                np.abs(
                    values_y
                    - ref[variable_y]
                )
            )

            j_gt = np.argmin(
                np.abs(
                    values_x
                    - ref[variable_x]
                )
            )

            D_groundtruth = (
                err_matrix[i_gt, j_gt]
            )

            print("-" * 50)

            if study_column is None:
                print(ref["label"])
            else:
                print(
                    f"{ref['label']} | "
                    f"{angle_symbol} = "
                    f"{angle_deg:.0f}°"
                )

            print("-" * 50)

            print("\nGlobal minimum:")
            print(variable_x, "=", x_best)
            print(variable_y, "=", y_best)

            print(
                distance_func_str,
                "=",
                D_min
            )

            print(
                "Number of global minima =",
                N_global_minima
            )

            print("\nEquivalent solutions:")
            print("Threshold =", threshold_abs)
            print("Number =", N_equivalent)

            print(
                "Percentage =",
                round(percentage, 2),
                "%"
            )

            print("\nExtent:")
            print(
                f"{variable_x}-width =",
                x_width
            )
            print(
                f"{variable_y}-width =",
                y_width
            )

            print("\nGround Truth:")
            print(
                variable_x,
                "=",
                ref[variable_x]
            )
            print(
                variable_y,
                "=",
                ref[variable_y]
            )
            print(
                distance_func_str,
                "=",
                D_groundtruth
            )

            # ----------------------------
            # Plot
            # ----------------------------

            fig, ax = plt.subplots(
                figsize=(8, 6)
            )

            im = ax.imshow(
                err_matrix,
                extent=[
                    values_x[0],
                    values_x[-1],
                    values_y[0],
                    values_y[-1]
                ],
                aspect="auto",
                origin="lower"
            )

            if (
                np.nanmin(err_matrix)
                <= threshold_abs
                <= np.nanmax(err_matrix)
            ):
                ax.contour(
                    values_x,
                    values_y,
                    err_matrix,
                    levels=[threshold_abs],
                    linewidths=2,
                    colors="red"
                )

            ax.scatter(
                ref[variable_x],
                ref[variable_y],
                marker="o",
                facecolors="none",
                edgecolors="white",
                s=40,
                label="Ground Truth"
            )

            # Achsen
            ax.set_xlabel(
                x_label,
                fontsize=label_fontsize
            )

            ax.set_ylabel(
                y_label,
                fontsize=label_fontsize
            )

            ax.tick_params(
                axis="both",
                labelsize=tick_fontsize
            )

            # Colorbar
            cbar = fig.colorbar(
                im,
                ax=ax
            )

            cbar.set_label(
                distance_func_str,
                fontsize=colorbar_fontsize
            )

            cbar.ax.tick_params(
                labelsize=tick_fontsize
            )

            # Titel
            if study_column is None:

                if (
                    theta_views is not None
                    and phi_views is not None
                    and len(theta_views) == 2
                    and len(phi_views) == 2
                ):

                    angle_info = (
                        f"View 1: θ = "
                        f"{np.rad2deg(theta_views[0]):.0f}°, "
                        f"φ = {np.rad2deg(phi_views[0]):.0f}° | "
                        f"View 2: θ = "
                        f"{np.rad2deg(theta_views[1]):.0f}°, "
                        f"φ = {np.rad2deg(phi_views[1]):.0f}°"
                    )

                else:
                    angle_info = ""

                if angle_info:
                    title = (
                        f"2D-Sensitivity Analysis: "
                        f"{variable_x} vs. {variable_y}\n"
                        f"{ref['label']}\n"
                        f"{angle_info}"
                    )
                else:
                    title = (
                        f"2D-Sensitivity Analysis: "
                        f"{variable_x} vs. {variable_y}\n"
                        f"{ref['label']}"
                    )

            else:
                title = (
                    f"2D-Sensitivity Analysis: "
                    f"{variable_x} vs. {variable_y}\n"
                    f"{ref['label']}, "
                    f"{angle_symbol} = "
                    f"{angle_deg:.0f}°"
                )

            ax.set_title(
                title,
                fontsize=title_fontsize
            )

            # Legende bleibt innerhalb der Grafik
            ax.legend(
                fontsize=legend_fontsize
            )

            fig.tight_layout()
            plt.show()

In [ ]:
def param_analysis_2d_two_views(ref_param, variable_x, values_x, variable_y, values_y, theta_views, phi_views, N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0, L0 = 1.0,
                                rho_f = 1.0, rho_r = 1.0, Ibg = 0.1, distance_func_str = "MSE", combined_error = "max", K = 5, sigma = 0, noise_switch = False,
                                poisson_scale = 1000, gaussian_std = 0.01, gamma_switch = False, gamma = 0.8, run_id = None, threshold_abs = 0.01,
                                save_csv = True, filename_prefix = "2d_sens_analysis_2views", filename_mode = "full", plot = True):
    
    """
    Performs a two-dimensional sensitivity analysis using two camera views.

    The two selected geometric parameters are varied over the specified
    value ranges while the remaining geometric parameter is kept at its
    reference value. For each parameter combination, simulated images from
    two camera views are compared with the corresponding reference images.
    The two view-specific errors are combined using the selected aggregation
    method.

    Parameters
    ----------
    ref_param : list of dict
        Reference cases containing the ground-truth parameters d, dx and dy.
        A label may additionally be provided for plotting.

    variable_x : {"d", "dx", "dy"}
        Geometric parameter varied along the x-axis.

    values_x : array-like
        Values of variable_x to be tested in mm.

    variable_y : {"d", "dx", "dy"}
        Geometric parameter varied along the y-axis.

    values_y : array-like
        Values of variable_y to be tested in mm.

    theta_views : iterable
        Polar viewing angles of the two camera views in radians.
        Exactly two values must be provided.

    phi_views : iterable
        Azimuth viewing angles of the two camera views in radians.
        Exactly two values must be provided.

    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the front
        and rear grids.

    Ibg : float
        Constant background intensity.

    distance_func_str : str
        Distance metric used to compare reference and candidate images,
        e.g. "MSE", "MAE" or "NMSE".

    combined_error : {"mean", "max"}
        Method used to combine the two view-specific image errors.

    K : int
        Number of best candidates requested internally from
        grid_search_topk. The returned DataFrame contains all
        evaluated parameter combinations.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, noise is applied to the generated reference images.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied.

    gamma : float
        Gamma exponent used for gamma correction.

    run_id : int, str or None
        Optional identifier included in the output filename.

    threshold_abs : float
        Absolute error threshold used by the 2D sensitivity plot to
        identify approximately equivalent solutions.

    save_csv : bool
        If True, the results are saved as a CSV file.

    filename_prefix : str
        Prefix used for the output CSV filename.

    filename_mode : {"full", "short"}
        Controls the amount of configuration information included
        in the output filename.

    plot : bool
        If True, the two-dimensional sensitivity results are plotted.

    Returns
    -------
    df : pandas.DataFrame
        Results of the two-view sensitivity analysis.

        The DataFrame contains the reference ID, tested geometric
        parameters d, dx and dy, the combined image error, and the
        individual view errors dm1 and dm2.

    Raises
    ------
    ValueError
        If variable_x or variable_y is not "d", "dx" or "dy", if both
        variables are identical, if theta_views or phi_views do not
        contain exactly two values, or if filename_mode is neither
        "full" nor "short".
    """
    
    if variable_x not in ("d", "dx", "dy"):
        raise ValueError("variable_x must be 'd', 'dx' or 'dy'")

    if variable_y not in ("d", "dx", "dy"):
        raise ValueError("variable_y must be 'd', 'dx' or 'dy'")

    if variable_x == variable_y:
        raise ValueError("variable_x and variable_y must be different.")

    if len(theta_views) != 2 or len(phi_views) != 2:
        raise ValueError("theta_views and phi_views must contain exactly two values.")

    if filename_mode not in ("full", "short"):
        raise ValueError("filename_mode must be 'full' or 'short'")

    theta_string = "-".join(str(np.rad2deg(theta)) for theta in theta_views)
    phi_string = "-".join(str(np.rad2deg(phi)) for phi in phi_views)


    if filename_mode == "full":
        filename = (
            f"{filename_prefix}_"
            f"var_{variable_x}_{variable_y}_"
            f"run_{run_id}_"
            f"N_{N}_"
            f"size_{size}_"
            f"p_{p}_"
            f"Df_{Df}_"
            f"Dr_{Dr}_"
            # f"L0_{L0}_"
            # f"rho_f_{rho_f}_"
            # f"rho_r_{rho_r}_"
            # f"Ibg_{Ibg}_"
            f"theta_{theta_string}_"
            f"phi_{phi_string}_"
            f"metric_{distance_func_str}_"
            f"combined_{combined_error}_"
            f"sigma_{sigma}_"
            f"noise_{noise_switch}_"
            f"poisson_{poisson_scale}_"
            f"gaussian_{gaussian_std}_"
            f"gamma_on_{gamma_switch}_"
            f"gamma_{gamma}.csv"
        )

    else:
        filename = (
            f"{filename_prefix}_"
            f"var_{variable_x}_{variable_y}_"
            f"run_{run_id}_"
            f"theta_{theta_string}_"
            f"phi_{phi_string}_"
            f"metric_{distance_func_str}_"
            f"combined_{combined_error}.csv"
        )

    all_results = []

    for ref in ref_param:
        ref_view1 = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                 d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = theta_views[0], phi = phi_views[0],
                                 sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale,
                                 gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")

        ref_view2 = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                 d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = theta_views[1], phi = phi_views[1],
                                 sigma = sigma, noise_switch = noise_switch, poisson_scale = poisson_scale,
                                 gaussian_std = gaussian_std, gamma_switch = gamma_switch, gamma = gamma, mode = "inverse")

        values = {
            "d": [ref["d"]],
            "dx": [ref["dx"]],
            "dy": [ref["dy"]]
        }

        values[variable_x] = values_x
        values[variable_y] = values_y

        d_values = values["d"]
        dx_values = values["dx"]
        dy_values = values["dy"]

        _, _, results = grid_search_topk(d_values, dx_values, dy_values, I_refs=[ref_view1["intensity"], ref_view2["intensity"]], N = N, size = size, p = p, Df = Df, Dr = Dr,
                                         L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg, thetas = [theta_views[0], theta_views[1]], phis = [phi_views[0], phi_views[1]],
                                         K = K, distance_func_str = distance_func_str, combined_error = combined_error, gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)

        all_results.append(results)

    if save_csv:
        df = save_results(all_results, filename = filename)

    else:
        history = []

        for ref_id, results in enumerate(all_results, start = 1):
            for result in results:
                history.append({
                    "ref_id": ref_id,
                    "d": result["d"],
                    "dx": result["dx"],
                    "dy": result["dy"],
                    "error": result["error"],
                    "dm1": result["dm1"],
                    "dm2": result["dm2"]
                })

        df = pd.DataFrame(history)

    if plot:
        plot_sensitivity_2d(df = df, ref_param = ref_param, variable_x = variable_x, values_x = values_x,
                                    variable_y = variable_y, values_y = values_y, p = p,
                                    distance_func_str = distance_func_str, threshold_abs = threshold_abs,
                                    theta_views = theta_views, phi_views = phi_views)

    return df

In [ ]:
def analyze_coarse_to_fine(df_all, df_top, ref_id = 1, K = 5, d_gt = None, dx_gt = None, dy_gt = None, p = 10.0,
                           distance_func_str = "MSE", theta_views = None, phi_views = None):
    
    """
    Analyzes and visualizes the results of a coarse-to-fine grid search.

    The function evaluates all tested parameter combinations and the stored
    top candidates for one reference case. It determines the minimum error
    per refinement level, identifies all parameter combinations that attain
    the minimum, optionally computes the distance to the ground-truth
    parameters, and visualizes the search progression and parameter space.

    Parameters
    ----------
    df_all : pandas.DataFrame
        DataFrame containing all parameter combinations evaluated during
        the coarse-to-fine search.

        Expected columns include:
        "ref_id", "level", "resolution_mm", "error",
        "d", "dx" and "dy".

    df_top : pandas.DataFrame
        DataFrame containing the stored top candidates of the
        coarse-to-fine search.

    ref_id : int
        Identifier of the reference case to be analyzed.

    K : int
        Number of top candidates displayed per refinement level.

    d_gt : float or None
        Ground-truth grid distance in mm. If d_gt, dx_gt and dy_gt are
        all provided, parameter errors relative to the ground truth are
        computed.

    dx_gt : float or None
        Ground-truth lateral displacement in x-direction in mm.

    dy_gt : float or None
        Ground-truth lateral displacement in y-direction in mm.

    p : float
        Grid period in mm. Used to compute periodic distances for dx
        and dy relative to the ground truth.

    distance_func_str : str
        Name of the distance metric used during the search,
        e.g. "MSE", "MAE" or "NMSE". Used for printed output,
        plot labels and colorbars.

    theta_views : iterable or None
        Viewing angles theta used for the evaluated camera views.
        The values are expected in degrees and are included in the
        printed output and plot titles if provided together with
        phi_views.

    phi_views : iterable or None
        Viewing angles phi used for the evaluated camera views.
        The values are expected in degrees and are included in the
        printed output and plot titles if provided together with
        theta_views.

    Returns
    -------
    best_per_level : pandas.DataFrame
        Parameter combinations attaining the minimum error at each
        refinement level. Multiple rows may be present for one level
        if several global minima exist.

        If d_gt, dx_gt and dy_gt are provided, the additional columns
        "d_error", "dx_error" and "dy_error" are included.

    df_all_ref : pandas.DataFrame
        All evaluated parameter combinations belonging to the selected
        reference case, sorted by refinement level and error.

    df_top_ref : pandas.DataFrame
        Stored top candidates belonging to the selected reference case,
        sorted by refinement level and error.

    Raises
    ------
    ValueError
        If theta_views and phi_views are both provided but do not have
        the same length.
    """

    LEVEL_COL = "level"
    RESOLUTION_COL = "resolution_mm"
    ERROR_COL = "error"
    REF_COL = "ref_id"
    D_COL = "d"
    DX_COL = "dx"
    DY_COL = "dy"


    if theta_views is not None and phi_views is not None:

        if len(theta_views) != len(phi_views):
            raise ValueError("theta_views and phi_views must have same length.")

        view_string = " | ".join(
            f"View {i + 1}: "
            f"θ = {theta:.0f}°, "
            f"φ = {phi:.0f}°"
            for i, (theta, phi) in enumerate(zip(theta_views, phi_views))
        )

    else:
        view_string = None



    print("-" * 50)
    print(f"Coarse to fine - Evaluation – Ref {ref_id}")
    
    if view_string is not None:
        print(view_string)

    print("-" * 50)

    print("\nShape of all datapoints:")
    print(df_all.shape)

    print("\nShape of all Top-K-datapoints:")
    print(df_top.shape)

    print("\nColumns:")
    print(df_all.columns.tolist())


    df_all_ref = df_all[df_all[REF_COL] == ref_id].copy()
    df_top_ref = df_top[df_top[REF_COL] == ref_id].copy()


    df_all_ref = df_all_ref.sort_values(by=[LEVEL_COL, ERROR_COL])
    df_top_ref = df_top_ref.sort_values(by=[LEVEL_COL, ERROR_COL])


    print("\n")
    print("-" * 50)
    print(f"TOP-{K} Results – Ref {ref_id}")
    print("-" * 50)

    for level in sorted(df_top_ref[LEVEL_COL].unique()):

        df_level = df_top_ref[df_top_ref[LEVEL_COL] == level].copy()
        df_level = (df_level.sort_values(ERROR_COL).head(K))
        resolution = df_level[RESOLUTION_COL].iloc[0]

        print(
            f"\nLevel {level} "
            f"(Step = {resolution} mm)"
        )

        print(
            df_level[
                [
                    D_COL,
                    DX_COL,
                    DY_COL,
                    ERROR_COL
                ]
            ].to_string(
                index = False,
                float_format = lambda x: f"{x:.6e}"
            )
        )


    min_error_per_level = (df_all_ref.groupby(LEVEL_COL)[ERROR_COL].min())
    best_per_level = df_all_ref[df_all_ref[ERROR_COL] == df_all_ref[LEVEL_COL]
                                .map(min_error_per_level)].copy()

    best_per_level = best_per_level.drop_duplicates(
        subset=[
            LEVEL_COL,
            D_COL,
            DX_COL,
            DY_COL,
            ERROR_COL
        ]
    )


    error_per_level = (df_all_ref.groupby(LEVEL_COL)[ERROR_COL].min().reset_index())
    
    print(f"\n{distance_func_str} per Level:")
    print(error_per_level)
    print("\n")
    print("-" * 50)
    print("Best result per level")
    print("-" * 50)

    print(
        best_per_level[
            [
                LEVEL_COL,
                RESOLUTION_COL,
                D_COL,
                DX_COL,
                DY_COL,
                ERROR_COL
            ]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.6e}"
        )
    )

    print("\n")
    print("-" * 50)
    print("Number of unique global minima per level")
    print("-" * 50)

    n_minima_per_level = (best_per_level.groupby(LEVEL_COL).size())

    print(n_minima_per_level)

    if all(v is not None for v in [d_gt, dx_gt, dy_gt]):

        best_per_level["d_error"] = (
            best_per_level[D_COL] - d_gt
        ).abs()

        best_per_level["dx_error"] = np.abs(
            ((best_per_level[DX_COL] - dx_gt + p / 2) % p)
            - p / 2
        )

        best_per_level["dy_error"] = np.abs(
            ((best_per_level[DY_COL] - dy_gt + p / 2) % p)
            - p / 2
        )

        print("\n")
        print("-" * 50)
        print("Distance to Ground Truth")
        print("-" * 50)

        print(
            best_per_level[
                [
                    LEVEL_COL,
                    D_COL,
                    DX_COL,
                    DY_COL,
                    "d_error",
                    "dx_error",
                    "dy_error",
                    ERROR_COL
                ]
            ].to_string(
                index=False,
                float_format=lambda x: f"{x:.6e}"
            )
        )

    levels = sorted(df_all_ref[LEVEL_COL].unique())


    plt.figure(
        figsize=(7, 5)
    )

    plt.plot(
        error_per_level[LEVEL_COL],
        error_per_level[ERROR_COL],
        marker="o",
        linewidth=2
    )

    plt.xlabel("Level")
    plt.ylabel(f"Best {distance_func_str}")

    plt.xticks(error_per_level[LEVEL_COL])


    plt.grid(
        True,
        alpha=0.3
    )

    title = (
        f"Trend of best {distance_func_str} – Ref {ref_id}"
    )

    if view_string is not None:
        title += f"\n{view_string}"

    plt.title(title)



    plt.tight_layout()
    plt.show()


    fig, axes = plt.subplots(
        3, 1,
        figsize = (7, 8),
        sharex=True
    )


    axes[0].scatter(
        best_per_level[LEVEL_COL],
        best_per_level[D_COL],
        s=30
    )

    axes[0].set_ylabel("d [mm]")
    axes[0].grid(
        True,
        alpha=0.3
    )


    axes[1].scatter(
        best_per_level[LEVEL_COL],
        best_per_level[DX_COL],
        s=30
    )

    axes[1].set_ylabel("dx [mm]")
    axes[1].grid(
        True,
        alpha=0.3
    )


    axes[2].scatter(
        best_per_level[LEVEL_COL],
        best_per_level[DY_COL],
        s=30
    )

    axes[2].set_ylabel("dy [mm]")
    axes[2].set_xlabel("Level")

    axes[2].grid(
        True,
        alpha=0.3
    )

    axes[2].set_xticks(levels)

    title = (f"Parameter distribution – Ref {ref_id}")

    if view_string is not None:
        title += f"\n{view_string}"

    fig.suptitle(title)

    plt.tight_layout()
    plt.show()


    def plot_search_space(x_col, y_col, x_label, y_label, title):

        fig, axes = plt.subplots(
            1, len(levels),
            figsize = (5 * len(levels), 4),
            squeeze = False
        )

        axes = axes[0]

        for ax, level in zip(axes, levels):

            df_level = df_all_ref[df_all_ref[LEVEL_COL] == level]


            scatter = ax.scatter(
                df_level[x_col],
                df_level[y_col],
                c=df_level[ERROR_COL],
                cmap="viridis",
                s=25
            )


            min_error = df_level[ERROR_COL].min()
            best = df_level[df_level[ERROR_COL] == min_error]

            ax.scatter(
                best[x_col],
                best[y_col],
                marker="x",
                s=50,
                linewidths=2,
                color="red",
                label="Bestes Ergebnis"
            )


            resolution = df_level[RESOLUTION_COL].iloc[0]

            ax.set_title(
                f"Level {level}\n"
                f"Delta = {resolution} mm"
            )

            ax.set_xlabel(x_label)
            ax.set_ylabel(y_label)

            ax.grid(
                True,
                alpha=0.2
            )

            ax.legend()


            fig.colorbar(
                scatter,
                ax=ax,
                label=distance_func_str
            )

        full_title = (
            f"{title} – Ref {ref_id}"
        )

        if view_string is not None:
            full_title += (
                f"\n{view_string}"
            )

        fig.suptitle(
            full_title,
            y=1.02
        )

        plt.tight_layout()
        plt.show()

    plot_search_space(
        x_col=DX_COL,
        y_col=D_COL,
        x_label="dx [mm]",
        y_label="d [mm]",
        title="Coarse to fine search: dx–d"
    )

    plot_search_space(
        x_col=DY_COL,
        y_col=D_COL,
        x_label="dy [mm]",
        y_label="d [mm]",
        title="Coarse to fine search: dy–d"
    )

    plot_search_space(
        x_col=DX_COL,
        y_col=DY_COL,
        x_label="dx [mm]",
        y_label="dy [mm]",
        title="Coarse to fine search: dx–dy"
    )

    return (
        best_per_level,
        df_all_ref,
        df_top_ref
    )

In [ ]:
def ctfs(ref_param, d_middle_start = 20.0, d_range_start = 10.0, resolutions = (2, 1, 0.5), theta_views = (45, ), phi_views = (0, ),
         N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0, L0 = 1.0, rho_f = 1.0, rho_r = 1.0, Ibg = 0.1, distance_func_str = "MSE",
         combined_error = "max", K = 5, sigma = 0, noise_switch = False, poisson_scale = 1000, gaussian_std = 0.01,
         gamma_switch = False, gamma = 0.8, run_id = None, save_csv = True, filename_mode = "full", plot = True):
        
    """
    Performs a coarse-to-fine grid search for one or two camera views.

    For each reference case, the function generates one reference image
    per camera view and searches for the geometric parameters d, dx and dy.
    The search starts from an initial parameter region and is refined over
    multiple resolution levels. At each level, the K best parameter
    combinations are used as centers of the search regions for the next
    refinement level.

    Parameters
    ----------
    ref_param : list of dict
        Reference cases containing the ground-truth parameters d, dx
        and dy.

    d_middle_start : float
        Initial center of the search interval for d in mm.

    d_range_start : float
        Initial half-width of the search interval for d in mm.

    resolutions : iterable
        Grid-search step sizes for the successive coarse-to-fine
        refinement levels in mm.

    theta_views : iterable
        Polar viewing angles of the camera views in degrees.
        One or two values must be provided.

    phi_views : iterable
        Azimuth viewing angles of the camera views in degrees.
        One or two values must be provided and the number of values
        must match theta_views.

    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the front
        and rear grids.

    Ibg : float
        Constant background intensity.

    distance_func_str : str
        Distance metric used to compare reference and candidate images,
        e.g. "MSE", "MAE" or "NMSE".

    combined_error : {"mean", "max"}
        Method used to combine the errors of two camera views.
        Ignored when only one view is used.

    K : int
        Number of best candidates retained from each search region
        and used to generate search regions for the next level.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, noise is applied to the generated reference images.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied.

    gamma : float
        Gamma exponent used for gamma correction.

    run_id : int, str or None
        Optional identifier stored in the result data and included
        in output filenames.

    save_csv : bool
        If True, the complete search history and the top-candidate
        history are saved as CSV files for each reference case.

    filename_mode : {"full", "short"}
        Controls the amount of configuration information included
        in the output filenames.

    plot : bool
        If True, the coarse-to-fine results are analyzed and
        visualized for each reference case.

    Returns
    -------
    df_results : pandas.DataFrame
        Combined search history for all reference cases.

        Each row represents one evaluated parameter combination and
        contains information including the run and reference IDs,
        candidate and parent IDs, refinement level, runtime, resolution,
        image errors, tested values of d, dx and dy, and the bounds and
        sampling information of the corresponding search region.

    Raises
    ------
    ValueError
        If no camera view is provided, if theta_views and phi_views
        have different lengths, if more than two camera views are
        specified, or if filename_mode is neither "full" nor "short".
    """

    if len(theta_views) == 0 or len(phi_views) == 0:
        raise ValueError("At least one camera view is required!")
    
    if len(theta_views) != len(phi_views):
        raise ValueError("theta_views and phi_views must have same length!")
    
    if len(theta_views) > 2 or len(phi_views) > 2:
        raise ValueError("theta_views and phi_views can't have more than 2 values!")

    if filename_mode not in ("full", "short"):
        raise ValueError("filename_mode must be 'full' or 'short'.")
    
    thetas = [np.deg2rad(theta) for theta in theta_views]
    phis = [np.deg2rad(phi) for phi in phi_views]

    
    theta_string =  "-".join(str(np.rad2deg(theta)) for theta in thetas)
    phi_string = "-".join(str(np.rad2deg(phi)) for phi in phis)
    resolution_string = "-".join(str(res)for res in resolutions)

    all_refs_history = []

    for ref_id, ref in enumerate(ref_param, start = 1):
        print("\n")
        print("-" * 50)
        print(f"Reference {ref_id}")
        print("-" * 50)

        if filename_mode == "full":

            filename_top = (
                f"top_history_"
                f"run{run_id}_"
                f"ref{ref_id}_"
                f"p{p}_"
                f"Df{Df}_"
                f"Dr{Dr}_"
                f"theta{theta_string}_"
                f"phi{phi_string}_"
                f"res{resolution_string}mm_"
                f"metric_{distance_func_str}_"
                f"combined_{combined_error}_"
                f"sigma_{sigma}_"
                f"noise_{noise_switch}_"
                f"poisson_{poisson_scale}_"
                f"gaussian_{gaussian_std}_"
                f"gamma_on_{gamma_switch}_"
                f"gamma_{gamma}.csv"
            )

            filename_all = (
                f"all_history_"
                f"run{run_id}_"
                f"ref{ref_id}_"
                f"p{p}_"
                f"Df{Df}_"
                f"Dr{Dr}_"
                f"theta{theta_string}_"
                f"phi{phi_string}_"
                f"res{resolution_string}mm_"
                f"metric_{distance_func_str}_"
                f"combined_{combined_error}_"
                f"sigma_{sigma}_"
                f"noise_{noise_switch}_"
                f"poisson_{poisson_scale}_"
                f"gaussian_{gaussian_std}_"
                f"gamma_on_{gamma_switch}_"
                f"gamma_{gamma}.csv"
            )

        else:

            filename_top = (
                f"top_history_"
                f"run{run_id}_"
                f"ref{ref_id}_"
                f"theta{theta_string}_"
                f"phi{phi_string}_"
                f"res{resolution_string}_"
                f"metric_{distance_func_str}_"
                f"combined_{combined_error}.csv"
            )

            filename_all = (
                f"all_history_"
                f"run{run_id}_"
                f"ref{ref_id}_"
                f"theta{theta_string}_"
                f"phi{phi_string}_"
                f"res{resolution_string}_"
                f"metric_{distance_func_str}_"
                f"combined_{combined_error}.csv"
            )

        I_refs = []
        
        for theta, phi in zip(thetas, phis):
            ref_view = simulation(N = N, size = size, p = p, Df = Df, Dr = Dr, L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg,
                                 d = ref["d"], dx = ref["dx"], dy = ref["dy"], theta = theta, phi = phi, sigma = sigma,
                                 noise_switch = noise_switch, poisson_scale = poisson_scale, gaussian_std = gaussian_std,
                                 gamma_switch = gamma_switch, gamma = gamma, mode = "simulation")

            I_refs.append(ref_view["intensity"])

        top_history = []
        all_history = []
        seen = set()
        candidate_id = 0

        candidates = [(
                candidate_id,
                -1,
                d_middle_start, d_range_start,
                p / 2, p / 2,
                p / 2, p / 2
            )]


        for level, resolution in enumerate(resolutions):
            precision = max(0, int(np.ceil(-np.log10(resolution))))

            print("-" * 50)
            print(f"LEVEL {level}")
            print(f"Resolution = {resolution} mm")
            print("-" * 50)

            new_candidates = []

            for (current_id, parent_id, d_middle, d_range, dx_middle, dx_range, dy_middle, dy_range) in candidates:
                step = resolution

                d_endpoint = int(np.ceil(d_range / step))
                d_values = np.round(d_middle + np.arange(-d_endpoint, d_endpoint + 1) * step, precision)
                d_values = np.unique(np.sort(d_values))

                dx_endpoint = int(np.ceil(dx_range / step))
                dx_values = np.round((dx_middle + np.arange(-dx_endpoint, dx_endpoint + 1) * step) % p, precision)
                dx_values = np.unique(np.sort(dx_values))

                dy_endpoint = int(np.ceil(dy_range / step))
                dy_values = np.round((dy_middle + np.arange(-dy_endpoint, dy_endpoint + 1) * step) % p, precision)
                dy_values = np.unique(np.sort(dy_values))

                num_templates = (len(d_values) * len(dx_values) * len(dy_values))

                start = time.perf_counter()

                best_k, _, all_results = grid_search_topk(d_values, dx_values, dy_values, I_refs = I_refs, N = N, size = size, p = p, Df = Df, Dr = Dr,
                                                                       L0 = L0, rho_f = rho_f, rho_r = rho_r, Ibg = Ibg, thetas = thetas, phis = phis, K = K,
                                                                       distance_func_str = distance_func_str, combined_error = combined_error,
                                                                       gamma_switch = gamma_switch, gamma = gamma, sigma = sigma)

                runtime = time.perf_counter() - start

                print(f"\nCandidate {current_id}")
                print(f"Templates : {num_templates}")
                print(f"Runtime   : {runtime:.3f} s")

                for result in all_results:
                    all_history.append({
                        "run": run_id,
                        "ref_id": ref_id,
                        "candidate_id": current_id,
                        "parent_id": parent_id,
                        "level": level,
                        "runtime_s": runtime,
                        "resolution_mm": resolution,

                        "error": result["error"],
                        "dm1": result.get("dm1", -1),
                        "dm2": result.get("dm2", -1),
                        "d": result["d"],
                        "dx": result["dx"],
                        "dy": result["dy"],

                        "d_middle": d_middle,
                        "d_range": d_range,
                        "d_samples": len(d_values),
                        "d_min": d_values.min(),
                        "d_max": d_values.max(),

                        "dx_middle": dx_middle,
                        "dx_range": dx_range,
                        "dx_samples": len(dx_values),
                        "dx_min": dx_values.min(),
                        "dx_max": dx_values.max(),

                        "dy_middle": dy_middle,
                        "dy_range": dy_range,
                        "dy_samples": len(dy_values),
                        "dy_min": dy_values.min(),
                        "dy_max": dy_values.max(),

                    })

                for rank, result in enumerate(best_k, start = 1):
                    err = result["error"]
                    d_best = result["d"]
                    dx_best = result["dx"]
                    dy_best = result["dy"]

                    entry = {
                        "run": run_id,
                        "ref_id": ref_id,
                        "candidate_id": current_id,
                        "parent_id": parent_id,
                        "child_id": -1,
                        "level": level,
                        "rank": rank,

                        "error": err,
                        "dm1": result.get("dm1", -1),
                        "dm2": result.get("dm2", -1),
                        "d": d_best,
                        "dx": dx_best,
                        "dy": dy_best,

                        "exact_match": err < 1e-12,
                        "runtime_s": runtime,
                        "resolution_mm": resolution,
                        "num_templates": num_templates,

                        "d_middle": d_middle,
                        "d_range": d_range,
                        "d_samples": len(d_values),
                        "d_min": d_values.min(),
                        "d_max": d_values.max(),

                        "dx_middle": dx_middle,
                        "dx_range": dx_range,
                        "dx_samples": len(dx_values),
                        "dx_min": dx_values.min(),
                        "dx_max": dx_values.max(),

                        "dy_middle": dy_middle,
                        "dy_range": dy_range,
                        "dy_samples": len(dy_values),
                        "dy_min": dy_values.min(),
                        "dy_max": dy_values.max(),

                    }

                    top_history.append(entry)

                    print(
                        f"--- Rank {rank}: ---"
                        f"err={err:.3e} "
                        f"d={d_best:.{precision}f} "
                        f"dx={dx_best:.{precision}f} "
                        f"dy={dy_best:.{precision}f}"
                    )

                    if level == len(resolutions) - 1:
                        continue

                    new_d_range = max(resolution, d_range / 2)
                    new_dx_range = max(resolution, dx_range / 2)
                    new_dy_range = max(resolution, dy_range / 2)

                    value = (
                        round(d_best, 6),
                        round(dx_best % p, 6),
                        round(dy_best % p, 6),
                        round(new_d_range, 6),
                        round(new_dx_range, 6),
                        round(new_dy_range, 6),
                    )

                    if value in seen:
                        continue

                    seen.add(value)
                    candidate_id += 1

                    child = (
                        candidate_id,
                        current_id,
                        d_best,
                        new_d_range,
                        dx_best % p,
                        new_dx_range,
                        dy_best % p,
                        new_dy_range
                    )

                    entry["child_id"] = candidate_id

                    new_candidates.append(child)

            candidates = new_candidates

        df_top_history = pd.DataFrame(top_history)
        df_all_history = pd.DataFrame(all_history)
        all_refs_history.append(df_all_history)

        if save_csv:
            df_top_history.to_csv(filename_top, index = False)
            df_all_history.to_csv(filename_all, index = False)

        if plot:
            analyze_coarse_to_fine(df_all = df_all_history, df_top = df_top_history, ref_id = ref_id, K = K,
                                   d_gt = ref["d"], dx_gt = ref["dx"], dy_gt = ref["dy"],
                                   p = p, distance_func_str = distance_func_str,
                                   theta_views = theta_views, phi_views = phi_views)
    df_results = pd.concat(
        all_refs_history,
        ignore_index=True
    )

    return df_results
    
    

In [ ]:
def evaluate_ctfs_runs(
    df_runs,
    ref_param,
    p=10.0
):
    
    """
    Evaluates multiple coarse-to-fine search runs.

    For each run and reference case, only the final refinement level is
    considered. The function identifies all distinct global minima at that
    level, computes their parameter errors relative to the corresponding
    ground-truth values, and checks whether the ground-truth solution is
    contained among the global minima.

    Parameters
    ----------
    df_runs : pandas.DataFrame
        Combined CTFS results from multiple runs.

        Expected columns include:
        "run", "ref_id", "level", "error", "d", "dx" and "dy".

    ref_param : list of dict
        Ground-truth reference cases containing the parameters d, dx and dy.

    p : float
        Grid period in mm. Used to compute periodic parameter errors for
        dx and dy.

    Returns
    -------
    df_summary : pandas.DataFrame
        Summary table containing one row per run and reference case.

        The DataFrame contains:
        "run", "ref_id", "final_level", "min_error",
        "n_global_minima", "gt_found", "d_minima",
        "dx_minima" and "dy_minima".

    df_minima : pandas.DataFrame
        Table containing all distinct global minima found at the final
        refinement level for all runs and reference cases.

        For each minimum, the DataFrame contains:
        "run", "ref_id", "d", "dx", "dy",
        "d_error", "dx_error", "dy_error" and "error".
    """

    summary_rows = []
    minima_rows = []

    # ============================================================
    # Runs
    # ============================================================

    for run_id in sorted(df_runs["run"].unique()):

        df_run = df_runs[
            df_runs["run"] == run_id
        ]

        # ========================================================
        # Referenzen
        # ========================================================

        for ref_id in sorted(df_run["ref_id"].unique()):

            ref = ref_param[ref_id - 1]

            df_ref = df_run[
                df_run["ref_id"] == ref_id
            ].copy()

            # ====================================================
            # Letztes CTFS-Level
            # ====================================================

            final_level = df_ref["level"].max()

            df_final = df_ref[
                df_ref["level"] == final_level
            ].copy()

            # ====================================================
            # Minimaler Fehler
            # ====================================================

            min_error = df_final["error"].min()

            mask_min = np.isclose(
                df_final["error"],
                min_error,
                rtol=1e-12,
                atol=1e-15
            )

            df_best = (
                df_final[mask_min]
                .drop_duplicates(
                    subset=[
                        "d",
                        "dx",
                        "dy"
                    ]
                )
                .copy()
            )

            # ====================================================
            # Abstand zur Ground Truth
            # ====================================================

            df_best["d_error"] = np.abs(
                df_best["d"] - ref["d"]
            )

            df_best["dx_error"] = np.abs(
                (
                    (
                        df_best["dx"]
                        - ref["dx"]
                        + p / 2
                    )
                    % p
                )
                - p / 2
            )

            df_best["dy_error"] = np.abs(
                (
                    (
                        df_best["dy"]
                        - ref["dy"]
                        + p / 2
                    )
                    % p
                )
                - p / 2
            )

            # ====================================================
            # Ground Truth unter globalen Minima?
            # ====================================================

            gt_mask = (
                np.isclose(
                    df_best["d"],
                    ref["d"]
                )
                &
                np.isclose(
                    df_best["dx_error"],
                    0
                )
                &
                np.isclose(
                    df_best["dy_error"],
                    0
                )
            )

            gt_found = bool(
                gt_mask.any()
            )

            # ====================================================
            # Alle globalen Minima
            # ====================================================

            for _, row in df_best.iterrows():

                minima_rows.append({
                    "run": run_id,
                    "ref_id": ref_id,

                    "d": row["d"],
                    "dx": row["dx"],
                    "dy": row["dy"],

                    "d_error": row["d_error"],
                    "dx_error": row["dx_error"],
                    "dy_error": row["dy_error"],

                    "error": row["error"]
                })

            # ====================================================
            # Zusammenfassung
            # ====================================================

            summary_rows.append({
                "run": run_id,
                "ref_id": ref_id,

                "final_level": final_level,
                "min_error": min_error,

                "n_global_minima": len(df_best),
                "gt_found": gt_found,

                "d_minima": ", ".join(
                    map(
                        str,
                        sorted(
                            df_best["d"].unique()
                        )
                    )
                ),

                "dx_minima": ", ".join(
                    map(
                        str,
                        sorted(
                            df_best["dx"].unique()
                        )
                    )
                ),

                "dy_minima": ", ".join(
                    map(
                        str,
                        sorted(
                            df_best["dy"].unique()
                        )
                    )
                )
            })

    df_summary = pd.DataFrame(
        summary_rows
    )

    df_minima = pd.DataFrame(
        minima_rows
    )

    return df_summary, df_minima

In [ ]:
def ctfs_multiple_runs(ref_param, n_runs = 10, d_middle_start = 20.0, d_range_start = 10.0, resolutions = (2, 1, 0.5), theta_views = (45,), phi_views = (0,),
                       N = 512, size = 100, p = 10.0, Df = 4.0, Dr = 5.0, L0 = 1.0, rho_f = 1.0, rho_r = 1.0, Ibg = 0.1, distance_func_str = "MSE",
                       combined_error="max", K = 5, sigma = 0, noise_switch = True, poisson_scale = 1000, gaussian_std = 0.01, gamma_switch = False,
                       gamma = 0.8, save_csv = False, save_comb = False, filename_mode = "full", plot = False):

    """
    Runs the coarse-to-fine search multiple times with an identical
    configuration and evaluates the combined results.

    Each run is executed with a unique run ID. The resulting search
    histories are concatenated into a single DataFrame and evaluated
    jointly using evaluate_ctfs_runs().

    Parameters
    ----------
    ref_param : list of dict
        Ground-truth reference cases containing the parameters d, dx
        and dy.

    n_runs : int
        Number of independent CTFS runs to perform.

    d_middle_start : float
        Initial center of the search interval for d in mm.

    d_range_start : float
        Initial half-width of the search interval for d in mm.

    resolutions : iterable
        Grid-search step sizes for the successive coarse-to-fine
        refinement levels in mm.

    theta_views : iterable
        Polar viewing angles of the camera views in degrees.
        One or two values may be provided.

    phi_views : iterable
        Azimuth viewing angles of the camera views in degrees.
        The number of values must match theta_views.

    N : int
        Image resolution in pixels along each axis.

    size : float
        Physical image size in mm along each axis.

    p : float
        Grid period in mm.

    Df : float
        Hole diameter of the front grid in mm.

    Dr : float
        Hole diameter of the rear grid in mm.

    L0 : float
        Illumination intensity scaling factor.

    rho_f, rho_r : float
        Intensity or reflectivity weighting factors of the front
        and rear grids.

    Ibg : float
        Constant background intensity.

    distance_func_str : str
        Distance metric used to compare reference and candidate images,
        e.g. "MSE", "MAE" or "NMSE".

    combined_error : {"mean", "max"}
        Method used to combine the errors of two camera views.
        Ignored when only one view is used.

    K : int
        Number of best candidates retained from each search region
        during the coarse-to-fine refinement.

    sigma : float
        Standard deviation of the Gaussian blur in pixels.

    noise_switch : bool
        If True, noise is applied to the generated reference images
        in each CTFS run.

    poisson_scale : float
        Scaling factor used for the Poisson noise model.

    gaussian_std : float
        Standard deviation of the additive Gaussian noise.

    gamma_switch : bool
        If True, gamma correction is applied.

    gamma : float
        Gamma exponent used for gamma correction.

    save_csv : bool
        If True, the individual CTFS result files are saved for each run
        and reference case.

    save_comb : bool
        If True, the combined results of all runs are saved as a single
        CSV file.

    filename_mode : {"full", "short"}
        Filename mode passed to ctfs() for the individual result files.

    plot : bool
        If True, the CTFS evaluation and plots are generated for each
        individual run.

    Returns
    -------
    df_runs : pandas.DataFrame
        Combined search history containing all evaluated parameter
        combinations from all runs and reference cases.

    df_summary : pandas.DataFrame
        Summary of the final CTFS level for each run and reference case,
        including the minimum error, number of global minima and whether
        the ground-truth solution was found.

    df_minima : pandas.DataFrame
        All distinct global minima found at the final refinement level
        across all runs and reference cases.

    Raises
    ------
    ValueError
        If n_runs is smaller than 1.
    """

    if n_runs < 1:
        raise ValueError("n_runs must be at least 1.")

    all_runs = []

    for run_id in range(1, n_runs + 1):

        print("\n")
        print("-" * 50)
        print(f"CTFS RUN {run_id} / {n_runs}")
        print("-" * 50)

        df_run = ctfs(
            ref_param=ref_param,
            d_middle_start=d_middle_start,
            d_range_start=d_range_start,
            resolutions=resolutions,
            theta_views=theta_views,
            phi_views=phi_views,
            N=N,
            size=size,
            p=p,
            Df=Df,
            Dr=Dr,
            L0=L0,
            rho_f=rho_f,
            rho_r=rho_r,
            Ibg=Ibg,
            distance_func_str=distance_func_str,
            combined_error=combined_error,
            K=K,
            sigma=sigma,
            noise_switch=noise_switch,
            poisson_scale=poisson_scale,
            gaussian_std=gaussian_std,
            gamma_switch=gamma_switch,
            gamma=gamma,
            run_id=run_id,
            save_csv=save_csv,
            filename_mode=filename_mode,
            plot=plot
        )

        all_runs.append(df_run)

    df_runs = pd.concat(all_runs, ignore_index = True)

    df_summary, df_minima = evaluate_ctfs_runs(
        df_runs = df_runs,
        ref_param = ref_param,
        p = p
    )

    theta_string =  "-".join(str((theta)) for theta in theta_views)
    phi_string = "-".join(str((phi)) for phi in phi_views)
    resolution_string = "-".join(str(res)for res in resolutions)

    filename_comb = (
        f"ctfs_combined_"
        f"runs{n_runs}_"
        f"theta{theta_string}_"
        f"phi{phi_string}_"
        f"res{resolution_string}.csv"
    )

    if save_comb:
        df_runs.to_csv(filename_comb, index = False)

    return (df_runs, df_summary, df_minima)

In [ ]:
def final_summarize_ctfs_statistics(df_summary, df_minima):

    """
    Summarizes statistical results from multiple coarse-to-fine search runs.

    For each reference case, the function distinguishes between runs with
    a unique global minimum and runs with multiple global minima. It
    computes success rates, statistics of the minimum image error, the
    average number of global minima, and parameter-error statistics for
    unique and ambiguous solutions.

    Parameters
    ----------
    df_summary : pandas.DataFrame
        Summary table generated by evaluate_ctfs_runs().

        Expected columns include:
        "ref_id", "run", "gt_found", "n_global_minima"
        and "min_error".

    df_minima : pandas.DataFrame
        Table containing the global minima of all runs.

        Expected columns include:
        "ref_id", "run", "d_error", "dx_error"
        and "dy_error".

    Returns
    -------
    df_stats : pandas.DataFrame
        Statistical summary with one row per reference case.

        The DataFrame contains:

        "ref_id" : int
            Reference case identifier.

        "n_runs" : int
            Number of evaluated runs.

        "n_unique_runs" : int
            Number of runs with exactly one global minimum.

        "gt_rate_percent" : float
            Percentage of runs in which the ground-truth solution
            is contained among the global minima.

        "unique_rate_percent" : float
            Percentage of runs with exactly one global minimum.

        "correct_unique_rate_percent" : float
            Percentage of all runs that have exactly one global minimum
            and for which this minimum corresponds to the ground truth.

        "mean_min_error", "std_min_error" : float
            Mean and standard deviation of the minimum image error
            across all runs.

        "mean_n_minima" : float
            Mean number of global minima across all runs.

        "mean_d_error_unique", "mean_dx_error_unique",
        "mean_dy_error_unique" : float
            Mean parameter errors for runs with a unique global minimum.

        "std_d_error_unique", "std_dx_error_unique",
        "std_dy_error_unique" : float
            Standard deviation of the parameter errors for runs with
            a unique global minimum.

        "mean_min_d_error_multiple",
        "mean_min_dx_error_multiple",
        "mean_min_dy_error_multiple" : float
            Mean of the smallest parameter error per run for runs
            with multiple global minima.

        "mean_max_d_error_multiple",
        "mean_max_dx_error_multiple",
        "mean_max_dy_error_multiple" : float
            Mean of the largest parameter error per run for runs
            with multiple global minima.
    """

    stats_rows = []

    error_cols = ["d_error", "dx_error", "dy_error"]

    for ref_id in sorted(df_summary["ref_id"].unique()):

        df_s = df_summary[
            df_summary["ref_id"] == ref_id
        ].copy()

        df_m = df_minima[
            df_minima["ref_id"] == ref_id
        ].copy()

        unique_mask = df_s["n_global_minima"] == 1
        multiple_mask = df_s["n_global_minima"] > 1

        unique_runs = df_s.loc[unique_mask, "run"]
        multiple_runs = df_s.loc[multiple_mask, "run"]

        df_unique = df_m[
            df_m["run"].isin(unique_runs)
        ].copy()

        df_multiple = df_m[
            df_m["run"].isin(multiple_runs)
        ].copy()

        row = {
            "ref_id": ref_id,
            "n_runs": len(df_s),
            "n_unique_runs": int(unique_mask.sum()),

            "gt_rate_percent":
                100 * df_s["gt_found"].mean(),

            "unique_rate_percent":
                100 * unique_mask.mean(),

            "correct_unique_rate_percent":
                100 * (unique_mask & df_s["gt_found"]).mean(),

            "mean_min_error":
                df_s["min_error"].mean(),

            "std_min_error":
                df_s["min_error"].std(ddof=1),

            "mean_n_minima":
                df_s["n_global_minima"].mean(),
        }

        for col in error_cols:

            row[f"mean_{col}_unique"] = (
                df_unique[col].mean()
            )

            row[f"std_{col}_unique"] = (
                df_unique[col].std(ddof=1)
            )

        for col in error_cols:

            if not df_multiple.empty:

                per_run = df_multiple.groupby("run")[col].agg(
                    min_error="min",
                    max_error="max"
                )

                row[f"mean_min_{col}_multiple"] = (
                    per_run["min_error"].mean()
                )

                row[f"mean_max_{col}_multiple"] = (
                    per_run["max_error"].mean()
                )

            else:
                row[f"mean_min_{col}_multiple"] = np.nan
                row[f"mean_max_{col}_multiple"] = np.nan

        stats_rows.append(row)

    return pd.DataFrame(stats_rows)

# Master Project

## Reference

In [ ]:
# grid parameters
p = 10.0
Df = 4.0
Dr = 5.0

# positions
d = 0.0 + 10.0 # Blechdicke
dx = 1.0
dy = 0.0

# angles
theta = np.deg2rad(45)
phi = np.deg2rad(0)

# size of simulation
N = 512
size = 10

# image parameters
L0 = 1.0
rho_f = 1.0
rho_r = 1.0
Ibg = 0.1

In [ ]:
ref_param = [
    {# Ref1
        "I": None,
        "d": 10.0,
        "dx": 1.0,
        "dy": 0.0,
        "theta": np.deg2rad(45),
        "phi": np.deg2rad(0),
        "label": "Ref1: dy = 0.0"
    },
    {# Ref2
        "I": None,
        "d": 10.0,
        "dx": 1.0,
        "dy": 2.0,
        "theta": np.deg2rad(45),
        "phi": np.deg2rad(0),
        "label": "Ref2: dy = 2.0"
    },
    {# Ref3
        "I": None,
        "d": 10.0,
        "dx": 1.0,
        "dy": 2.0,
        "theta": np.deg2rad(45),
        "phi": np.deg2rad(45),
        "label": "Ref3: dy = 2.0, phi = 45°"
    },
    {# Ref4
        "I": None,
        "d": 10.0,
        "dx": 1.0,
        "dy": 2.0,
        "theta": np.deg2rad(30),
        "phi": np.deg2rad(0),
        "label": "Ref4: dy = 2.0, theta = 30°"
    }
]

In [ ]:
for ref in ref_param:
    ref["I"] = simulation(
        d=ref["d"],
        dx=ref["dx"],
        dy=ref["dy"],
        theta=ref["theta"],
        phi=ref["phi"],
        mode="simulation"
    )

In [ ]:
d_values = np.linspace(10, 50, 100)
d_values = np.unique(
    np.append(d_values, [ref["d"] for ref in ref_param])
)
dx_values = np.linspace(0, 3*p, 100)
dx_values = np.unique(
    np.append(dx_values, [ref["dx"] for ref in ref_param])
)
dy_values = np.linspace(0, 3*p, 100)
dy_values = np.unique(
    np.append(dy_values, [ref["dy"] for ref in ref_param])
)

## Test: Consistency

In [ ]:
# Concistancy test angle
consistancy_test_angle = np.deg2rad(0)

# Consistancy test distance
consistancy_test_distance = [10.0, 15.0, 20.0, 25.0, 30.0]

In [ ]:
d_ref = d
dx_ref = dx
dy_ref = dy
theta_ref = consistancy_test_angle

result = simulation(d = d_ref, dx = dx_ref, dy = dy_ref, theta = theta_ref, mode = "simulation")

sv = result["sv"]

assert np.allclose(sv, [0.0, 0.0]), f"Test failed: sv = {sv} instead of [0, 0]"

print("Test 1 passed: theta = 0 -> sv = 0")

In [ ]:
sv_norms = []

for d in consistancy_test_distance:

    result = simulation(d = d, theta = np.deg2rad(45), phi = np.deg2rad(0), mode = "inverse"
    )
    print(result["sv"])
    sv_norms.append(np.linalg.norm(result["sv"]))

sv_norms = np.array(sv_norms)

assert np.all(np.diff(sv_norms) >= 0), \
    "Test failed: ||sv|| does not grow monotonically with d"

print("Test 2 passed: ||sv|| grows monotonically with d")

## 1-dimensional sensitivity analysis

### Influence of d

In [ ]:
df_d = param_analysis_1d(
    ref_param=ref_param,
    variable="d",
    values=d_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

#### MAE

In [ ]:
df_d = param_analysis_1d(
    ref_param=ref_param,
    variable="d",
    values=d_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MAE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

#### NMSE

In [ ]:
df_d = param_analysis_1d(
    ref_param=ref_param,
    variable="d",
    values=d_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="NMSE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

### Influence of dx

#### MSE

In [ ]:
df_dx = param_analysis_1d(
    ref_param=ref_param,
    variable="dx",
    values=dx_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

#### MAE

In [ ]:
df_dx = param_analysis_1d(
    ref_param=ref_param,
    variable="dx",
    values=dx_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MAE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

#### NMSE

In [ ]:
df_dx = param_analysis_1d(
    ref_param=ref_param,
    variable="dx",
    values=dx_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="NMSE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

### Influence of dy

#### MSE

In [ ]:
df_dy = param_analysis_1d(
    ref_param=ref_param,
    variable="dy",
    values=dy_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

#### MAE

In [ ]:
df_dy = param_analysis_1d(
    ref_param=ref_param,
    variable="dy",
    values=dy_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MAE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

#### NMSE

In [ ]:
df_dy = param_analysis_1d(
    ref_param=ref_param,
    variable="dy",
    values=dy_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="NMSE",
    K=5,
    sigma=0,
    noise_switch=False,
    run_id=1,
    filename_mode="short"
)

## 2-dimensional sensitivity analysis

### Variables: d and dx

In [ ]:
df_d_dx = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dx",
    values_x=dx_values,
    variable_y="d",
    values_y=d_values,
    distance_func_str="MSE",
    sigma=0,
    noise_switch=False,
    threshold_abs=0.01,
    filename_mode="short"
)

### Variables: d and dy

In [ ]:
df_d_dy = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dy",
    values_x=dy_values,
    variable_y="d",
    values_y=d_values,
    distance_func_str="MSE",
    sigma=0,
    noise_switch=False,
    threshold_abs=0.01,
    filename_mode="short"
)

### Variables: dx and dy

In [ ]:
df_dx_dy = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dx",
    values_x=dx_values,
    variable_y="dy",
    values_y=dy_values,
    distance_func_str="MSE",
    sigma=0,
    noise_switch=False,
    threshold_abs=0.01,
    filename_mode="short"
)

## Different angles

In [ ]:
angles_study = [np.deg2rad(0), np.deg2rad(30),
               np.deg2rad(45), np.deg2rad(60)]
angles_study_sym = [np.deg2rad(-45), np.deg2rad(45)]


### Theta

#### 1-dimensional sensitivity analysis

##### Influence of d

In [ ]:
df_d = param_analysis_1d(
    ref_param=ref_param,
    variable="d",
    values=d_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    theta_study = angles_study,
    K=5,
    filename_prefix = "1d_sens_analysis",
    filename_mode="short"
)

###### Symmetry with negative angle

In [ ]:
df_d_sym = param_analysis_1d(
    ref_param=ref_param,
    variable="d",
    values=d_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    theta_study = angles_study_sym,
    K=5,
    filename_prefix = "1d_sens_analysis_sym",
    filename_mode="short"
)

##### Influence of dx

In [ ]:
df_dx = param_analysis_1d(
    ref_param=ref_param,
    variable="dx",
    values=dx_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    theta_study = angles_study,
    K=5,
    filename_prefix = "1d_sens_analysis",
    filename_mode="short"
)

###### Symmetry with negative angle

In [ ]:
df_dx_sym = param_analysis_1d(
    ref_param=ref_param,
    variable="dx",
    values=dx_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    theta_study = angles_study_sym,
    K=5,
    filename_prefix = "1d_sens_analysis_sym",
    filename_mode="short"
)

##### Influence of dy

In [ ]:
df_dy = param_analysis_1d(
    ref_param=ref_param,
    variable="dy",
    values=dy_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    theta_study = angles_study,
    K=5,
    filename_prefix = "1d_sens_analysis",
    filename_mode="short"
)

###### Symmetry with negative angle

In [ ]:
df_dy_sym = param_analysis_1d(
    ref_param=ref_param,
    variable="dy",
    values=dy_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    theta_study = angles_study_sym,
    K=5,
    filename_prefix = "1d_sens_analysis_sym",
    filename_mode="short"
)

#### 2-dimensional sensitivity analysis

In [ ]:
angles_2d = [np.deg2rad(-45), np.deg2rad(0), np.deg2rad(45), np.deg2rad(60)]

##### Variables: d and dx

In [ ]:
df_d_dx = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dx",
    values_x=dx_values,
    variable_y="d",
    values_y=d_values,
    distance_func_str="MSE",
    theta_study = angles_2d,
    threshold_abs=0.01,
    filename_mode="short"
)

##### Variables: d and dy

In [ ]:
df_d_dy = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dy",
    values_x=dy_values,
    variable_y="d",
    values_y=d_values,
    distance_func_str="MSE",
    theta_study = angles_2d,
    threshold_abs=0.01,
    filename_mode="short"
)

##### Variables: dx and dy

In [ ]:
df_dy_dx = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dx",
    values_x=dx_values,
    variable_y="dy",
    values_y=dy_values,
    distance_func_str="MSE",
    theta_study = angles_2d,
    threshold_abs=0.01,
    filename_mode="short"
)

### Phi

In [ ]:
phi_angles = [np.deg2rad(0), np.deg2rad(30), np.deg2rad(45)
              , np.deg2rad(60), np.deg2rad(90)]

#### 1-dimensional sensitivity analysis

##### Influence of d

In [ ]:
df_d = param_analysis_1d(
    ref_param=ref_param,
    variable="d",
    values=d_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    phi_study = phi_angles,
    K=5,
    filename_prefix = "1d_sens_analysis",
    filename_mode="short"
)

##### Influence of dx

In [ ]:
df_dx = param_analysis_1d(
    ref_param=ref_param,
    variable="dx",
    values=dx_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    phi_study = phi_angles,
    K=5,
    filename_prefix = "1d_sens_analysis",
    filename_mode="short"
)

##### Influence of dy

In [ ]:
df_dy = param_analysis_1d(
    ref_param=ref_param,
    variable="dy",
    values=dy_values,
    p=p,
    Df=Df,
    Dr=Dr,
    distance_func_str="MSE",
    phi_study = phi_angles,
    K=5,
    filename_prefix = "1d_sens_analysis",
    filename_mode="short"
)

#### 2-dimensional sensitivity analysis

In [ ]:
phi_angles_2d = [np.deg2rad(0), np.deg2rad(30), np.deg2rad(45), np.deg2rad(90)]

##### Variables: d and dx

In [ ]:
df_d_dx = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dx",
    values_x=dx_values,
    variable_y="d",
    values_y=d_values,
    distance_func_str="MSE",
    phi_study = phi_angles_2d,
    threshold_abs=0.01,
    filename_mode="short"
)

##### Variables: d and dy

In [ ]:
df_d_dy = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dy",
    values_x=dy_values,
    variable_y="d",
    values_y=d_values,
    distance_func_str="MSE",
    phi_study = phi_angles_2d,
    threshold_abs=0.01,
    filename_mode="short"
)

##### Variables: dx and dy

In [ ]:
df_dy_dx = param_analysis_2d(
    ref_param=ref_param,
    p=p,
    Df=Df,
    Dr=Dr,
    variable_x="dx",
    values_x=dx_values,
    variable_y="dy",
    values_y=dy_values,
    distance_func_str="MSE",
    phi_study = phi_angles_2d,
    threshold_abs=0.01,
    filename_mode="short"
)

## 2nd Image

In [ ]:
ref_param_2d = ref_param[:2].copy()

### dx-d

#### Mean

In [ ]:
df_d_dx_2views = param_analysis_2d_two_views(
    ref_param=ref_param_2d,
    variable_x="dx",
    values_x=dx_values,
    variable_y="d",
    values_y=d_values,
    theta_views=[
        np.deg2rad(45),
        np.deg2rad(45)
    ],

    phi_views=[
        np.deg2rad(0),
        np.deg2rad(90)
    ],
    p=p,
    Df=Df,
    Dr=Dr,
    combined_error="mean",
    threshold_abs=0.01
)

#### Max

In [ ]:
df_dx_d_2views = param_analysis_2d_two_views(
    ref_param=ref_param_2d,
    variable_x="dx",
    values_x=dx_values,
    variable_y="d",
    values_y=d_values,
    theta_views=[
        np.deg2rad(45),
        np.deg2rad(45)
    ],
    phi_views=[
        np.deg2rad(0),
        np.deg2rad(90)
    ],
    p=p,
    Df=Df,
    Dr=Dr,
    combined_error="max",
    threshold_abs=0.01
)

### dy-d

#### Mean

In [ ]:
df_d_dy_2views = param_analysis_2d_two_views(
    ref_param=ref_param_2d,
    variable_x="dy",
    values_x=dy_values,
    variable_y="d",
    values_y=d_values,
    theta_views=[
        np.deg2rad(45),
        np.deg2rad(45)
    ],
    phi_views=[
        np.deg2rad(0),
        np.deg2rad(90)
    ],
    p=p,
    Df=Df,
    Dr=Dr,
    combined_error="mean",
    threshold_abs=0.01
)

#### Max

In [ ]:
df_d_dy_2views = param_analysis_2d_two_views(
    ref_param=ref_param_2d,
    variable_x="dy",
    values_x=dy_values,
    variable_y="d",
    values_y=d_values,
    theta_views=[
        np.deg2rad(45),
        np.deg2rad(45)
    ],
    phi_views=[
        np.deg2rad(0),
        np.deg2rad(90)
    ],
    p=p,
    Df=Df,
    Dr=Dr,
    combined_error="max",
    threshold_abs=0.01
)

### dx-dy

#### Mean

In [ ]:
df_dy_dx_2views = param_analysis_2d_two_views(
    ref_param=ref_param_2d,
    variable_x="dx",
    values_x=dx_values,
    variable_y="dy",
    values_y=dy_values,
    theta_views=[
        np.deg2rad(45),
        np.deg2rad(45)
    ],
    phi_views=[
        np.deg2rad(0),
        np.deg2rad(90)
    ],
    p=p,
    Df=Df,
    Dr=Dr,
    combined_error="mean",
    threshold_abs=0.01
)

#### Max

In [ ]:
df_dy_dx_2views = param_analysis_2d_two_views(
    ref_param=ref_param_2d,
    variable_x="dx",
    values_x=dx_values,
    variable_y="dy",
    values_y=dy_values,
    theta_views=[
        np.deg2rad(45),
        np.deg2rad(45)
    ],
    phi_views=[
        np.deg2rad(0),
        np.deg2rad(90)
    ],
    p=p,
    Df=Df,
    Dr=Dr,
    combined_error="max",
    threshold_abs=0.01
)

## Coarse To Fine Search

### CTFS Basetest

In [ ]:
ctfs(
    ref_param=ref_param,
    d_middle_start=20.0,
    d_range_start=10.0,
    resolutions=(2, 1, 0.5),
    theta_views=(45,),
    phi_views=(0,),
    N=512,
    size=100,
    p=10.0,
    Df=4.0,
    Dr=5.0,
    distance_func_str="MSE",
    combined_error="max",
    K=5,
    sigma=0,
    noise_switch=False,
    gamma_switch=False,
    run_id=1,
    save_csv=True,
    filename_mode="full",
    plot=True
)

### CTFS 2 Views

In [ ]:
ctfs(
    ref_param=ref_param,
    d_middle_start=20.0,
    d_range_start=10.0,
    resolutions=(2, 1, 0.5),
    theta_views=(45,45),
    phi_views=(0,90),
    N=512,
    size=100,
    p=10.0,
    Df=4.0,
    Dr=5.0,
    distance_func_str="MSE",
    combined_error="max",
    K=5,
    sigma=0,
    noise_switch=False,
    gamma_switch=False,

    run_id=1,
    save_csv=True,
    filename_mode="full",
    plot=True
)

### CTFS Control Case

In [ ]:
ctfs(
    ref_param=ref_param,
    d_middle_start=20.0,
    d_range_start=10.0,
    resolutions=(2, 1, 0.5),
    theta_views=(45,30),
    phi_views=(0,90),
    N=512,
    size=100,
    p=10.0,
    Df=4.0,
    Dr=5.0,
    distance_func_str="MSE",
    combined_error="max",
    K=5,
    sigma=0,
    noise_switch=False,
    gamma_switch=False,
    run_id=1,
    save_csv=True,
    filename_mode="full",
    plot=True
)

### CTFS with sigma

In [ ]:
ctfs(
    ref_param=ref_param,
    d_middle_start=20.0,
    d_range_start=10.0,
    resolutions=(2, 1, 0.5),
    theta_views=(45,45),
    phi_views=(0,90),
    N=512,
    size=100,
    p=10.0,
    Df=4.0,
    Dr=5.0,
    distance_func_str="MSE",
    combined_error="max",
    K=5,
    sigma=1,
    noise_switch=False,
    gamma_switch=False,
    poisson_scale = 1000,
    gaussian_std = 0.01,
    run_id=1,
    save_csv=True,
    filename_mode="full",
    plot=True
)

### CTFS with noise

In [ ]:
ctfs(
    ref_param=ref_param,
    d_middle_start=20.0,
    d_range_start=10.0,
    resolutions=(2, 1, 0.5),
    theta_views=(45,45),
    phi_views=(0,90),
    N=512,
    size=100,
    p=10.0,
    Df=4.0,
    Dr=5.0,
    distance_func_str="MSE",
    combined_error="max",
    K=5,
    sigma=1,
    noise_switch=True,
    gamma_switch=False,
    poisson_scale = 1000,
    gaussian_std = 0.01,
    run_id=1,
    save_csv=True,
    filename_mode="full",
    plot=True
)

### k runs of CTFS with noise

#### Multiple minima

In [ ]:
df_runs, df_summary, df_minima = ctfs_multiple_runs(
    ref_param=ref_param,
    n_runs=10,
    d_middle_start=20.0,
    d_range_start=10.0,
    resolutions=(2, 1, 0.5),
    theta_views=(45,45),
    phi_views=(0,90),
    N=512,
    size=100,
    p=10.0,
    Df=4.0,
    Dr=5.0,
    distance_func_str="MSE",
    combined_error="max",
    K=5,
    sigma=1,
    noise_switch=True,
    poisson_scale=1000,
    gaussian_std=0.01,
    gamma_switch=False,
    gamma=0.8,
    save_csv=False,
    save_comb=True,
    filename_mode="short",
    plot=False
)
final_summarize_ctfs_statistics(df_summary, df_minima)

#### Unique minimum

In [ ]:
df_runs, df_summary, df_minima = ctfs_multiple_runs(
    ref_param=ref_param,
    n_runs=10,
    d_middle_start=20.0,
    d_range_start=10.0,
    resolutions=(2, 1, 0.5),
    theta_views=(45,30),
    phi_views=(0,90),
    N=512,
    size=100,
    p=10.0,
    Df=4.0,
    Dr=5.0,
    distance_func_str="MSE",
    combined_error="max",
    K=5,
    sigma=1,
    noise_switch=True,
    poisson_scale=1000,
    gaussian_std=0.01,
    gamma_switch=False,
    gamma=0.8,
    save_csv=False,
    save_comb=True,
    filename_mode="short",
    plot=False
)
final_summarize_ctfs_statistics(df_summary, df_minima)